In [1]:
import io
import numpy as np
import scipy as sci
import scipy.constants as scic
import scipy.integrate as scii
import scipy.signal as scs
import scipy.constants as scc
from functools import cmp_to_key
from scipy.interpolate import CubicSpline as cubichers
from scipy.interpolate import interp1d
from typing import Union, Any, List
from dataclasses import dataclass, replace
from dataclasses import field as dcfield
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, Label
from ipywidgets import VBox, interactive_output, IntSlider,fixed, IntText, Dropdown, RadioButtons,interact_manual, Layout, Button, FloatText,Output, HBox
from IPython.display import display, Image
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, FigureCanvasAgg
from matplotlib.backends.backend_tkagg import NavigationToolbar2Tk
from matplotlib.ticker import FormatStrFormatter,EngFormatter
from pandas import DataFrame, concat, read_csv
import warnings
from copy import deepcopy
import jax as jx
import jax.numpy as jxn
from chex import dataclass as jaxdatclass
import optax
import jax.nn as jnn
import jax.scipy.signal as jsig
import jax.scipy.special as jsp
from numba import njit
from functools import partial
from scipy.optimize import least_squares as leasts
from scipy.stats import linregress
import concurrent.futures
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
import re
import threading
from itertools import product as iterproduct
from epraya.base_cris import *
from epraya.base_ham import *
from epraya.base_plot import *
from epraya.base_powd import *
from epraya.base_rotate import *
import sys

stopvar=threading.Event()
result={}

@dataclass
class Fitresult:
    '''
    Container for the results of the Fitting function.
    
    Returns
    -------
    
    Ham : object
        Hamiltonian configured with the fitted parameters
    spc : np.array
        Fitted spectrum
    params : np.array
        Values of the fitted parameters.
    method : str
        Method used in the fit.
    chi2 : float
        Chi square of the fit.
    redchi2 : float
        Chi square divided by (Number of point - Number of parameters) for the calculation of variance.
    residuals : np.array
        Residual array of the fit.
    denochi : float
        Denominator of the redchi2.
    message : str
        Message of the final state of the fit.
    variance : np.array
        Covariance matrix of the fit.
    paramerrors : np.array
        Variance array of the fitted parameters.
    paramerrorsdict : dict
        Dictionary to relate the variance array with the parameters labels.
    paramlabels : list
        Labels of the fitted parameters.
    success : bool
        If the process was succesfull in fitting the parameters with the number of iterations.
    iterations : int
        Number of iterations for the fit.
    '''
    Ham : object
    spc : np.ndarray
    params : np.ndarray
    method : str
    chi2 : float
    redchi2 : float
    residuals : np.ndarray
    denochi : int #Denominator for the  calculation of the residuals from chi2
    message : str=''
    variance : np.ndarray=None
    paramerrors : np.ndarray=None
    paramerrorsdict:dict=None
    paramlabels:list=None 
    success : bool=True
    iterations : int=None
    history: np.ndarray=None
    
def Jacobian(fx,x0,args=(),eps=1e-6,bounds=None):
    '''
    Function for the calculation of the numerical jacobian using central finite difference.
    '''
    x0=np.asarray(x0,float)
    f0=fx(x0,*args)
    J=np.zeros((len(f0),len(x0)))
    for i in range(len(x0)):
        h=eps*max(1.0,abs(x0[i]))
        xp=x0.copy()
        xm=x0.copy()
        xp[i]+=h
        xm[i]-=h
        if bounds is not None:
            xp[i]=min(xp[i],bounds[1][i])
            xm[i]=max(xm[i],bounds[0][i])
        if xp[i]==xm[i]:
            continue
        f1=fx(xp,*args)
        f2=fx(xm,*args)
        J[:,i]=(f1-f2)/(xp[i]-xm[i])
    return J


def Toconsideset(point,baixo,alto,J,residuals,tol=1e-6):
    span=alto-baixo
    fixed=span<=2e-4
    frac=(point-baixo)/np.where(span>0,span,1.0)
    #Get chi2/2 derivative respect to parameters
    grad=J.T@residuals                      # d(chi2/2)/dx
    atlo=(~fixed)&(frac<=tol)&(grad>0.0)
    athi=(~fixed)&(frac>=1.0-tol)&(grad<0.0)
    return fixed|atlo|athi

#Establish the difference between multisystems and single systems.
def findtypeham(Ham2):
    fitfunctions={Hval: dict(unpack=UnpackHam,res=Residuals,pack=Packtoscipy),Multham: dict(unpack=UnpackHam2,res=Residuals2,pack=Packtoscipy2)}
    return fitfunctions.get(type(Ham2))

def Formatvar(Ham,Var):
    if isinstance(Ham,Hval):
        labels=[]
        if Var.g!=0.0:
            labels+=['gx','gy','gz']
        if Var.A!=0.0:
            labels+=['Ax','Ay','Az']
        if Var.Q!=0.0:
            labels+=['Qx','Qy','Qz']
        if Var.D!=0.0:
            labels+=['D','E']
        if np.any(Var.Hpp):
            labels+=['Hppg','Hppl']
        return labels
    else:
        labels=[]
        for ar in range(len(Var.Mvary)):
            vmi=Var.Mvary[ar]
            if vmi.g!=0.0:
                labels+=[f'gx_{ar+1}',f'gy_{ar+1}',f'gz_{ar+1}']
            if vmi.A!=0.0:
                labels+=[f'Ax_{ar+1}',f'Ay_{ar+1}',f'Az_{ar+1}']
            if vmi.Q!=0.0:
                labels+=[f'Qx_{ar+1}',f'Qy_{ar+1}',f'Qz_{ar+1}']
            if vmi.D!=0.0:
                labels+=[f'D_{ar+1}',f'E_{ar+1}']
            if np.any(vmi.Hpp):
                labels+=[f'Hppg_{ar+1}',f'Hppl_{ar+1}']
        return labels
    
def Formaterrors(perr,labels):
    if perr is None:
        return None
    parame=[]
    for n,v in zip(labels,perr):
        if np.isnan(v):
            parame.append(f"{n}: Undefined")
        else:
            parame.append(f"{n}: {v:.4g}")
    return " | ".join(parame)

def Buildres(bestpoint,Ham1,Exp,exper,Vary,functiona,Mr,method,J=None,success=True,message='',iterations=None,dattype='data'):
    back=findtypeham(Ham1)
    if back is None:
        raise TypeError(f'Wrong Hamiltoinan type: {type(Ham1)}.')
    unpack=back['unpack']
    resfu=back['res']
    _,baixo,alto=back['pack'](Ham1,Vary)
    besHam=unpack(bestpoint,Ham1,Vary)
    if functiona in ['Powder','Calpowder']:
        functiona='Powder'
    residuals=resfu(bestpoint,Ham1,Exp,exper,Vary,functiona,Mr,dattype)
    if functiona in ['Powder']:
        spect=Powder(besHam,Exp,M=Mr,graph=False)[1]
    elif functiona in ['Eresonant']:
        spect=Eresonant(besHam,Exp,graph=False,table=False)[1]
    elif functiona in ['Mulpol']:
        spect=Mulpol(besHam,Exp,M=Mr,graph=False)[1]
    elif functiona in ['Music']:
        spect=Music(besHam,Exp,graph=False,table=False)[1]
    labels=Formatvar(Ham1,Vary)
    if J is None:
        J=Jacobian(resfu,bestpoint,args=(Ham1,Exp,exper,Vary,functiona,Mr,dattype),bounds=(baixo,alto))
    active=Toconsideset(np.asarray(bestpoint,float),baixo,alto,J,residuals)
    free=~active
    n,p=len(residuals),len(bestpoint)
    deno=max(n-int(free.sum()),1)
    chi2=float(np.sum(residuals**2))
    redchi2=chi2/deno
    perr=np.full(p,np.nan) 
    variance=np.full((p,p),np.nan)
    if free.any():
        varf,perrf,_=Selectvarian(J[:,free],redchi2)
        perr[free]=perrf
        variance[np.ix_(free,free)]=varf
    return Fitresult(Ham=besHam,spc=spect,params=bestpoint,method=method,chi2=chi2,redchi2=redchi2,residuals=residuals,denochi=deno,variance=variance,paramerrors=perr,paramlabels=labels,success=success,message=message,iterations=iterations)


import time

class TargetReached(Exception): 
    pass

class BudgetReached(Exception):
    pass
    
class Tracker:
    '''
    Best cost in function of the number of evaluations of the spectrum. Calculates n saves, cost, time, paramters in every iterations.
    '''
    budget=None
    def __init__(self): 
        self.reset()
    def reset(self):
        self.n,self.best,self.hist,self.t0=0,np.inf,[],time.perf_counter()
    target=None      
    def record(self,cost,xfun=None):
        self.n+=1
        #DOesn't improve
        if cost<self.best:                                 
            self.best=cost
            if xfun is None:
                x=[] 
            else:
                x=list(np.asarray(xfun(),float))
            self.hist.append([self.n,cost,time.perf_counter()-self.t0]+x)
        if self.target is not None and cost<=self.target:
            raise TargetReached
        if self.budget is not None and self.n >= self.budget:
            raise BudgetReached
    def array(self):
        h=np.array(self.hist,float)
        if not len(h):
            return h
        #Last row is the total
        return np.vstack([h,np.r_[self.n,self.best,time.perf_counter()-self.t0,h[-1,3:]]])
        
TRACK=Tracker()
    
def CostfNM(exper,intens,metric='L2'):
    if metric=='rmse':
        # Root Mean Squared Error
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        error=np.sqrt(np.mean((intens-exper)**2))
        return error
    elif metric=='correlation':
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        corr=np.corrcoef(exper,intens)
        pearson=corr[0, 1]
        if np.isnan(pearson):
            return 1e6
        return 1-pearson
    elif metric=='L2':
        normi=np.linalg.norm(intens)
        if normi==0 or np.isnan(normi):
            return 1e6
        norme=np.linalg.norm(exper)
        enorm=exper/norme
        inorm=intens/normi
        return np.sum((inorm-enorm)**2)

def Nelder1(Hamer,Expe,Vara,exper,eps=1e-10,maximal=5000,datype='data',mode='p',seed=451,Mr=70):
    '''
    Implementation of the Nelder Mead algorithm for fitting data for a single system.

    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of iterations to evalue the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    spce : np.array
        Best adjusted spectrum using Powder or Eresonant functions.
    '''
    TRACK.reset()
    #Based on the code by Hadrien Crassous
    def Fincost(points,funcname):
        if stopvar.is_set():
            return 1e6
        jl=0
        if Var.g!=0.0:
            gt=points[jl:jl+3]
            if (gt[0]<Var.g[0] or gt[0]>Var.g[1] or gt[1]<Var.g[2] or gt[1]>Var.g[3] or
            gt[2]<Var.g[4] or gt[2]>Var.g[5]):
                return 1e6
            Ham.g=gt
            jl+=3
        if Var.A!=0.0:
            At=points[jl:jl+3]
            if (At[0]<Var.A[0] or At[0]>Var.A[1] or At[1]<Var.A[2] or At[1]>Var.A[3] or
            At[2]<Var.A[4] or At[2]>Var.A[5]):
                return 1e6
            Ham.A=At
            jl+=3
        if Var.Q!=0.0:
            Qt=points[jl:jl+3]
            if (Qt[0]<Var.Q[0] or Qt[0]>Var.Q[1] or Qt[1]<Var.Q[2] or Qt[1]>Var.Q[3] or
            Qt[2]<Var.Q[4] or Qt[2]>Var.Q[5]):
                return 1e6
            Ham.Q=Qt
            jl+=3
        if Var.D!=0.0:
            Dt=points[jl:jl+2]
            if (Dt[0]<Var.D[0] or Dt[0]>Var.D[1] or Dt[1]<Var.D[2] or Dt[1]>Var.D[3]):
                return 1e6
            Ham.D=Dt
            jl+=2
        if np.any(Var.Hpp):
            Htp=points[jl:jl+2]
            if (Htp[0]<Var.Hpp[0] or Htp[0]>Var.Hpp[1] or Htp[1]<Var.Hpp[2] or Htp[1]>Var.Hpp[3]):
                return 1e6
            Ham.Hpp=Htp
        if funcname in ['Powder']:
            fieldae,spect=Powder(Ham,Exp,M=Mr,graph=False)
        elif funcname in ['Eresonant']:
            fieldae,spect=Eresonant(Ham,Exp,graph=False,table=False)
        if datype=='integral':
            spect=scii.cumulative_trapezoid(spect,fieldae,initial=0)
        wcost=CostfNM(exper,spect)
        TRACK.record(wcost,lambda: np.array(points,float))
        return wcost
    Ham=deepcopy(Hamer)
    Exp=deepcopy(Expe)
    Var=deepcopy(Vara)
    initexper=exper
    if mode=='p':
        funtiona=Powder#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Eresonant#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    pointx=[]
    stp=[]
    lowfron=[]
    hifron=[]
    if datype=='data':
        exper=exper
    elif datype=='integral':
        fielda=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    else:
        raise ValueError(f"Data type {datype} is not supported. Use integral or data instead.")
    if Var.g!=0.0:
        pointx.extend(Ham.g)
        stp.extend([(Var.g[1]-Var.g[0])/20,(Var.g[3]-Var.g[2])/20,(Var.g[5]-Var.g[4])/20])
        lowfron.extend([Var.g[0],Var.g[2],Var.g[4]])
        hifron.extend([Var.g[1],Var.g[3],Var.g[5]])
    if Var.A!=0.0:
        pointx.extend(Ham.A)
        stp.extend([(Var.A[1]-Var.A[0])/20,(Var.A[3]-Var.A[2])/20,(Var.A[5]-Var.A[4])/20])
        lowfron.extend([Var.A[0],Var.A[2],Var.A[4]])
        hifron.extend([Var.A[1],Var.A[3],Var.A[5]])
    if Var.Q!=0.0:
        pointx.extend(Ham.Q)
        stp.extend([(Var.Q[1]-Var.Q[0])/20,(Var.Q[3]-Var.Q[2])/20,(Var.Q[5]-Var.Q[4])/20])
        lowfron.extend([Var.Q[0],Var.Q[2],Var.Q[4]])
        hifron.extend([Var.Q[1],Var.Q[3],Var.Q[5]])
    if Var.D!=0.0:
        pointx.extend(Ham.D)
        stp.extend([(Var.D[1]-Var.D[0])/20,(Var.D[3]-Var.D[2])/20])
        lowfron.extend([Var.D[0],Var.D[2]])
        hifron.extend([Var.D[1],Var.D[3]])
    if np.any(Var.Hpp):
        pointx.extend(Ham.Hpp)
        stp.extend([(Var.Hpp[1]-Var.Hpp[0])/20,(Var.Hpp[3]-Var.Hpp[2])/20])
        lowfron.extend([Var.Hpp[0],Var.Hpp[2]])
        hifron.extend([Var.Hpp[1],Var.Hpp[3]])
    pointx=np.array(pointx,dtype=float)
    stp=np.array(stp,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    nparams=len(pointx)
    simplex=np.zeros((nparams+1,nparams))
    price=np.zeros(nparams+1)
    simplex[0]=pointx
    price[0]=Fincost(pointx,funcname)
    melhor=20
    #Iteration parameters
    alpha=1
    gamma=1+(2/nparams)
    rho=0.75-(1/(2*nparams))
    sigma=1-(1/nparams)
    np.random.seed(int(seed))
    for la in range(0,nparams):
        npoint=np.copy(pointx)
        change=np.random.uniform(-1,1)
        npoint[la]+=change*stp[la]
        simplex[la+1]=npoint
        price[la+1]=Fincost(npoint,funcname)
    itera=0
    restarts=0
    mrestarts=5
    try:
        while itera<maximal:
            if stopvar.is_set():
                print("Process stopped by user.")
                stopvar.clear()
                break
            simplex=simplex[np.argsort(price)]
            price=price[np.argsort(price)]
            center=np.mean(simplex[:-1],axis=0)
            melhor=simplex[0]
            segunda=simplex[-2]
            pior=simplex[-1]
            cmelhor=price[0]
            csegunda=price[-2]
            cpior=price[-1]
            if (cpior-cmelhor)<eps:
                if restarts<mrestarts:
                    restarts+=1
                    print(f"Restarting around the best points...")
                    pointx=np.copy(simplex[0])
                    for la in range(1,len(simplex)-2):
                        rad=np.random.uniform(lowfron,hifron)
                        simplex[la]=rad
                        price[la]=Fincost(rad,funcname)
                    continue
                else:
                    print("Stopped the process.")
                    break
            if itera%10==0:
                print(f"Iteration: {itera} | Best cost: {cmelhor:.5e}")
                if Var.g!=0.0:
                    print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
                if Var.A!=0.0:
                    print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
                if Var.Q!=0.0:
                    print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
                if Var.D!=0.0:
                    print(f'D={Ham.D[0]} | E={Ham.D[1]}')
                if np.any(Var.Hpp):
                    print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
            vect=np.random.normal(0,1,nparams)
            vnor=np.linalg.norm(vect)
            dist=np.abs(pior-melhor)
            pecenter=center+0.1*(vect*dist)
            #Reflection
            ref=pecenter+alpha*(pecenter-pior)
            cref=Fincost(ref,funcname)
            if cmelhor<=cref<csegunda:
                simplex[-1]=ref
                price[-1]=cref
            elif cref<cmelhor:
                # Expansion
                expan=pecenter+gamma*(ref-pecenter)
                cexpan=Fincost(expan,funcname)
                if cexpan<cref:
                    simplex[-1]=expan
                    price[-1]=cexpan
                else:
                    simplex[-1]=ref
                    price[-1]=cref
            else:
                #Contraction
                if cref<cpior:
                    contra=center+rho*(ref-center) #External
                else:
                    contra=center+rho*(pior-center) #Internal
                ccontra=Fincost(contra,funcname)
                if ccontra<min(cref,cpior):
                    simplex[-1]=contra
                    price[-1]=ccontra
                else:
                    # Shrink
                    for ila in range(1,len(simplex)):
                        simplex[ila]=melhor+sigma*(simplex[ila]-melhor)
                        price[ila]=Fincost(simplex[ila],funcname)
            itera+=1
    except KeyboardInterrupt:
        _=Fincost(melhor,funcname)
        print("\n"+"="*50)
        print(f"Process stopped at iteration: {itera} with best cost: {cmelhor:.5e}")
        print("="*50)
        if Var.g!=0.0:
            print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
        if Var.A!=0.0:
            print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
        if Var.Q!=0.0:
            print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
        if Var.D!=0.0:
            print(f'D={Ham.D[0]} | E={Ham.D[1]}')
        if np.any(Var.Hpp):
            print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
        fitresult=Buildres(melhor,Ham,Exp,initexper,Var,funcname,Mr,method='Nelder-Mead',success=False,message='Nelder-Mead stopped by user',iterations=itera,dattype=datype)
        return fitresult.Ham,fitresult
    _=Fincost(melhor,funcname)
    print("\n"+"="*50)
    print(f"Process stopped at iteration: {itera} with best cost: {cmelhor:.5e}")
    print("="*50)
    if Var.g!=0.0:
        print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
    if Var.A!=0.0:
        print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
    if Var.Q!=0.0:
        print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
    if Var.D!=0.0:
        print(f'D={Ham.D[0]} | E={Ham.D[1]}')
    if np.any(Var.Hpp):
        print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
    fitresult=Buildres(melhor,Ham,Exp,initexper,Var,funcname,Mr,method='Nelder-Mead',success=(itera<maximal),message='Converged with Nelder-Mead',iterations=itera,dattype=datype)
    return fitresult.Ham,fitresult

def Nelder2(Hamer,Expe,Vara,exper,eps=1e-10,maximal=5000,datype='data',mode='p',seed=451,Mr=70):
    '''
    Implementation of the Nelder Mead algorithm for fitting data for multisystems.

    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of iterations to evalue the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution.
    Mr : int
        Number of divisions for the Delaunay grid.    
        
    Returns
    -------
    spce : np.array
        Best adjusted spectrum using Mulpol or Music functions.
    '''
    TRACK.reset()
    #Based on the code by Hadrien Crassous
    def Fincost(points,funcname):
        if stopvar.is_set():
            return 1e6
        jl=0
        for i in range(len(Ham.Mulham)):
            if Var.Mvary[i].g!=0.0:
                gt=points[jl:jl+3]
                if (gt[0]<Var.Mvary[i].g[0] or gt[0]>Var.Mvary[i].g[1] or gt[1]<Var.Mvary[i].g[2] or gt[1]>Var.Mvary[i].g[3] or gt[2]<Var.Mvary[i].g[4] or gt[2]>Var.Mvary[i].g[5]):
                    return 1e6
                Ham.Mulham[i].g=gt
                jl+=3
            if Var.Mvary[i].A!=0.0:
                At=points[jl:jl+3]
                if (At[0]<Var.Mvary[i].A[0] or At[0]>Var.Mvary[i].A[1] or At[1]<Var.Mvary[i].A[2] or At[1]>Var.Mvary[i].A[3] or At[2]<Var.Mvary[i].A[4] or At[2]>Var.Mvary[i].A[5]):
                    return 1e6
                Ham.Mulham[i].A=At
                jl+=3
            if Var.Mvary[i].Q!=0.0:
                Qt=points[jl:jl+3]
                if (Qt[0]<Var.Mvary[i].Q[0] or Qt[0]>Var.Mvary[i].Q[1] or Qt[1]<Var.Mvary[i].Q[2] or Qt[1]>Var.Mvary[i].Q[3] or Qt[2]<Var.Mvary[i].Q[4] or Qt[2]>Var.Mvary[i].Q[5]):
                    return 1e6
                Ham.Mulham[i].Q=Qt
                jl+=3
            if Var.Mvary[i].D!=0.0:
                Dt=points[jl:jl+2]
                if (Dt[0]<Var.Mvary[i].D[0] or Dt[0]>Var.Mvary[i].D[1] or Dt[1]<Var.Mvary[i].D[2] or Dt[1]>Var.Mvary[i].D[3]):
                    return 1e6
                Ham.Mulham[i].D=Dt
                jl+=2
            if np.any(Var.Mvary[i].Hpp):
                Htp=points[jl:jl+2]
                if (Htp[0]<Var.Mvary[i].Hpp[0] or Htp[0]>Var.Mvary[i].Hpp[1] or Htp[1]<Var.Mvary[i].Hpp[2] or Htp[1]>Var.Mvary[i].Hpp[3]):
                    return 1e6
                Ham.Mulham[i].Hpp=Htp
                jl+=2
        try:
            if funcname in ['Mulpol']:
                fieldae,spect=funtiona(Ham,Exp,M=Mr,graph=False)
            elif funcname in ['Music']:
                fieldae,spect=funtiona(Ham,Exp,graph=False,table=False)
            if datype=='integral':
                spect=scii.cumulative_trapezoid(spect,fieldae,initial=0)
            wcost=CostfNM(exper,spect)
            TRACK.record(wcost,lambda: np.array(points,float))
            return wcost
        except Exception:
            return 1e6
    Ham=deepcopy(Hamer)
    Exp=deepcopy(Expe)
    Var=deepcopy(Vara)
    initexper=exper
    if mode=='p':
        funtiona=Mulpol#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Music#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    pointx=[]
    stp=[]
    lowfron=[]
    hifron=[]
    if datype=='data':
        exper=exper
    elif datype=='integral':
        fielda=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    else:
        raise ValueError(f"Data type {datype} is not supported. Use integral or data instead.")
    numberes=len(Ham.Mulham)
    for lke in range(0,numberes):
        if Var.Mvary[lke].g!=0.0:
            pointx.extend(Ham.Mulham[lke].g)
            stp.extend([(Var.Mvary[lke].g[1]-Var.Mvary[lke].g[0])/20,(Var.Mvary[lke].g[3]-Var.Mvary[lke].g[2])/20,(Var.Mvary[lke].g[5]-Var.Mvary[lke].g[4])/20])
            lowfron.extend([Var.Mvary[lke].g[0],Var.Mvary[lke].g[2],Var.Mvary[lke].g[4]])
            hifron.extend([Var.Mvary[lke].g[1],Var.Mvary[lke].g[3],Var.Mvary[lke].g[5]])
        if Var.Mvary[lke].A!=0.0:
            pointx.extend(Ham.Mulham[lke].A)
            stp.extend([(Var.Mvary[lke].A[1]-Var.Mvary[lke].A[0])/20,(Var.Mvary[lke].A[3]-Var.Mvary[lke].A[2])/20,(Var.Mvary[lke].A[5]-Var.Mvary[lke].A[4])/20])
            lowfron.extend([Var.Mvary[lke].A[0],Var.Mvary[lke].A[2],Var.Mvary[lke].A[4]])
            hifron.extend([Var.Mvary[lke].A[1],Var.Mvary[lke].A[3],Var.Mvary[lke].A[5]])
        if Var.Mvary[lke].Q!=0.0:
            pointx.extend(Ham.Mulham[lke].Q)
            stp.extend([(Var.Mvary[lke].Q[1]-Var.Mvary[lke].Q[0])/20,(Var.Mvary[lke].Q[3]-Var.Mvary[lke].Q[2])/20,(Var.Mvary[lke].Q[5]-Var.Mvary[lke].Q[4])/20])
            lowfron.extend([Var.Mvary[lke].Q[0],Var.Mvary[lke].Q[2],Var.Mvary[lke].Q[4]])
            hifron.extend([Var.Mvary[lke].Q[1],Var.Mvary[lke].Q[3],Var.Mvary[lke].Q[5]])
        if Var.Mvary[lke].D!=0.0:
            pointx.extend(Ham.Mulham[lke].D)
            stp.extend([(Var.Mvary[lke].D[1]-Var.Mvary[lke].D[0])/20,(Var.Mvary[lke].D[3]-Var.Mvary[lke].D[2])/20])
            lowfron.extend([Var.Mvary[lke].D[0],Var.Mvary[lke].D[2]])
            hifron.extend([Var.Mvary[lke].D[1],Var.Mvary[lke].D[3]])
        if np.any(Var.Mvary[lke].Hpp):
            pointx.extend(Ham.Mulham[lke].Hpp)
            stp.extend([(Var.Mvary[lke].Hpp[1]-Var.Mvary[lke].Hpp[0])/20,(Var.Mvary[lke].Hpp[3]-Var.Mvary[lke].Hpp[2])/20])
            lowfron.extend([Var.Mvary[lke].Hpp[0],Var.Mvary[lke].Hpp[2]])
            hifron.extend([Var.Mvary[lke].Hpp[1],Var.Mvary[lke].Hpp[3]])
    pointx=np.array(pointx,dtype=float)
    stp=np.array(stp,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    nparams=len(pointx)
    simplex=np.zeros((nparams+1,nparams))
    price=np.zeros(nparams+1)
    simplex[0]=pointx
    price[0]=Fincost(pointx,funcname)

    #Iteration parameters
    alpha=1
    gamma=1+(2/nparams)
    rho=0.75-(1/(2*nparams))
    sigma=1-(1/nparams)
    np.random.seed(int(seed))
    for la in range(0,nparams):
        npoint=np.copy(pointx)
        change=np.random.uniform(-1,1)
        npoint[la]+=change*stp[la]
        simplex[la+1]=npoint
        price[la+1]=Fincost(npoint,funcname)
    itera=0
    melhor=20
    restarts=0
    mrestarts=5
    try:
        while itera<maximal:
            if stopvar.is_set():
                print("Process stopped by user.")
                stopvar.clear()
                break    
            simplex=simplex[np.argsort(price)]
            price=price[np.argsort(price)]
            center=np.mean(simplex[:-1],axis=0)
            melhor=simplex[0]
            segunda=simplex[-2]
            pior=simplex[-1]
            cmelhor=price[0]
            csegunda=price[-2]
            cpior=price[-1]
            if (cpior-cmelhor)<eps:
                if restarts<mrestarts:
                    restarts+=1
                    print(f"Restarting around the best points...")
                    pointx=np.copy(simplex[0])
                    for la in range(1,len(simplex)-2):
                        rad=np.random.uniform(lowfron,hifron)
                        simplex[la]=rad
                        price[la]=Fincost(rad,funcname)
                    continue
                else:
                    print("Stopped the process.")
                    break
            if itera%10==0:
                print(f"Iteration: {itera} | Best cost: {cmelhor:.5e}")
                for i in range(len(Ham.Mulham)):
                    print(f"--- System {i+1} ---")
                    if Var.Mvary[i].g!=0.0:
                        print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
                    if Var.Mvary[i].A!=0.0:
                        print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
                    if Var.Mvary[i].Q!=0.0:
                        print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
                    if Var.Mvary[i].D!=0.0:
                        print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
                    if np.any(Var.Mvary[i].Hpp):
                        print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
            vect=np.random.normal(0,1,nparams)
            vnor=np.linalg.norm(vect)
            dist=np.abs(pior-melhor)
            pecenter=center+0.1*(vect*dist)
            #Reflection
            ref=pecenter+alpha*(pecenter-pior)
            cref=Fincost(ref,funcname)
            if cmelhor<=cref<csegunda:
                simplex[-1]=ref
                price[-1]=cref
            elif cref<cmelhor:
                # Expansion
                expan=pecenter+gamma*(ref-pecenter)
                cexpan=Fincost(expan,funcname)
                if cexpan<cref:
                    simplex[-1]=expan
                    price[-1]=cexpan
                else:
                    simplex[-1]=ref
                    price[-1]=cref
            else:
                #Contraction
                if cref<cpior:
                    contra=center+rho*(ref-center) #External
                else:
                    contra=center+rho*(pior-center) #Internal
                ccontra=Fincost(contra,funcname)
                if ccontra<min(cref,cpior):
                    simplex[-1]=contra
                    price[-1]=ccontra
                else:
                    # Shrink
                    for ila in range(1,len(simplex)):
                        simplex[ila]=melhor+sigma*(simplex[ila]-melhor)
                        price[ila]=Fincost(simplex[ila],funcname)
            itera+=1
    except KeyboardInterrupt:
        _=Fincost(melhor,funcname)
        print("\n"+"="*50)
        print(f"Process stopped at iteration: {itera} with best cost: {cmelhor:.5e}")
        print("="*50)
        for i in range(len(Ham.Mulham)):
            print(f"--- System {i+1} ---")
            if Var.Mvary[i].g!=0.0:
                print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
            if Var.Mvary[i].A!=0.0:
                print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
            if Var.Mvary[i].Q!=0.0:
                print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
            if Var.Mvary[i].D!=0.0:
                print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
            if np.any(Var.Mvary[i].Hpp):
                print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
        fitresult=Buildres(melhor,Ham,Exp,initexper,Var,funcname,Mr,method='Nelder-Mead',success=False,message='Nelder-Mead stopped by user',iterations=itera,dattype=datype)
        return fitresult.Ham,fitresult
    _=Fincost(melhor,funcname)
    print("\n"+"="*50)
    print(f"Process stopped at iteration: {itera} with best cost: {cmelhor:.5e}")
    print("="*50)
    for i in range(len(Ham.Mulham)):
        print(f"--- System {i+1} ---")
        if Var.Mvary[i].g!=0.0:
            print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
        if Var.Mvary[i].A!=0.0:
            print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
        if Var.Mvary[i].Q!=0.0:
            print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
        if Var.Mvary[i].D!=0.0:
            print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
        if np.any(Var.Mvary[i].Hpp):
            print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
    fitresult=Buildres(melhor,Ham,Exp,initexper,Var,funcname,Mr,method='Nelder-Mead',success=(itera<maximal),message='Converged with Nelder-Mead',iterations=itera,dattype=datype)
    return fitresult.Ham,fitresult

def Nelder(Hamer,Expe,Vara,exper,eps=1e-10,maximal=5000,datype='data',mode='p',seed=451,Mr=70):
    '''
    Implementation of the Nelder Mead algorithm for fitting data, based on the code by Hadrien Crassous. Uses the 4 classic possibilities reflection, expansion, contraction and shrink with their respective constants, alpha, gamma, rho and sigma, defined depending in the number of parameters to vary (nparams). With that, their definitions are:
    
    alpha=1
    
    gamma=1+(2/nparams)
    
    rho=0.75-(1/(2*nparams))
    
    sigma=1-(1/nparams)
    
    The algorithm also has a restart function, if the points converge to a local minimun and a heat up function, changing one of the parameters in order to vary the cost.
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of iterations to evalue the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution.
    Mr : int
        Number of divisions for the Delaunay grid.    
    
    Returns
    -------
    Ham : Class
        Container for the fitted hamiltonian parameters.
    fitresult : Class 
        Container for the results of the Fitting function.
    '''

    if type(Hamer)==Multham:
         if Expe.Mexp[0].Points!=len(exper):
            Expe.Mexp[0].Points=len(exper)
         return Nelder2(Hamer,Expe,Vara,exper,eps,maximal,datype,mode,seed,Mr)
    elif type(Hamer)==Hval:
         if Expe.Points!=len(exper):
            Expe.Points=len(exper)
         return Nelder1(Hamer,Expe,Vara,exper,eps,maximal,datype,mode,seed,Mr)

def CostfG(exper,intens,metric='L2'):
    if metric=='rmse':
        # Root Mean Squared Error
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        error=np.sqrt(np.mean((intens-exper)**2))
        return error
    elif metric=='correlation':
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        corr=np.corrcoef(exper,intens)
        pearson=corr[0, 1]
        if np.isnan(pearson):
            return 1e6
        return 1-pearson
    elif metric=='L2':
        normi=np.linalg.norm(intens)
        if normi==0 or np.isnan(normi):
            return 1e6
        norme=np.linalg.norm(exper)
        enorm=exper/norme
        inorm=intens/normi
        return np.sum((inorm-enorm)**2)
        
def Cross(vat1,vat2):
    ret=len(vat1)
    mask=np.random.randint(0,2,size=ret)
    son1=np.zeros(ret)
    son2=np.zeros(ret)
    for iw in range(0,ret):
        if mask[iw]==1:
            son1[iw]=vat1[iw]
            son2[iw]=vat2[iw]
        else:
            son1[iw]=vat2[iw]
            son2[iw]=vat1[iw]
    return son1,son2

def Mutategauss(fela,lowfron,hifron,proba=0.1,desv=0.05):
    nparams=len(fela)
    mutant=np.copy(fela)
    for ap in range(0,nparams):
        if np.random.rand()<proba:
            rangee=hifron[ap]-lowfron[ap]
            sigma=desv*rangee
            noise=np.random.normal(0,sigma)
            mutant[ap]+=noise
            if mutant[ap]<lowfron[ap]:
                mutant[ap]=lowfron[ap]
            elif mutant[ap]>hifron[ap]:
                mutant[ap]=hifron[ap]
    return mutant

def Genio1(Hamer,Expe,Vara,exper,eps=1e-10,maximal=30,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting function for the experimental data using the genetic algorithm for a single system.
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of generations to evalue.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    spce : np.array
        Best adjusted spectrum using the Powder or Eresonant functions.
    '''
    def Fincost(points,funcname):
        if stopvar.is_set():
            return 1e6
        jl=0
        if Var.g!=0.0:
            gt=points[jl:jl+3]
            if (gt[0]<Var.g[0] or gt[0]>Var.g[1] or gt[1]<Var.g[2] or gt[1]>Var.g[3] or
            gt[2]<Var.g[4] or gt[2]>Var.g[5]):
                return 1e6
            Ham.g=gt
            jl+=3
        if Var.A!=0.0:
            At=points[jl:jl+3]
            if (At[0]<Var.A[0] or At[0]>Var.A[1] or At[1]<Var.A[2] or At[1]>Var.A[3] or
            At[2]<Var.A[4] or At[2]>Var.A[5]):
                return 1e6
            Ham.A=At
            jl+=3
        if Var.Q!=0.0:
            Qt=points[jl:jl+3]
            if (Qt[0]<Var.Q[0] or Qt[0]>Var.Q[1] or Qt[1]<Var.Q[2] or Qt[1]>Var.Q[3] or
            Qt[2]<Var.Q[4] or Qt[2]>Var.Q[5]):
                return 1e6
            Ham.Q=Qt
            jl+=3
        if Var.D!=0.0:
            Dt=points[jl:jl+2]
            if (Dt[0]<Var.D[0] or Dt[0]>Var.D[1] or Dt[1]<Var.D[2] or Dt[1]>Var.D[3]):
                return 1e6
            Ham.D=Dt
            jl+=2
        if np.any(Var.Hpp):
            Htp=points[jl:jl+2]
            if (Htp[0]<Var.Hpp[0] or Htp[0]>Var.Hpp[1] or Htp[1]<Var.Hpp[2] or Htp[1]>Var.Hpp[3]):
                return 1e6
            Ham.Hpp=Htp
        if funcname in ['Powder']:
            fieldae,spect=Powder(Ham,Exp,M=Mr,graph=False)
        elif funcname in ['Eresonant']:
            fieldae,spect=Eresonant(Ham,Exp,graph=False,table=False)
        if datype=='integral':
            spect=scii.cumulative_trapezoid(spect,fieldae,initial=0)
        wcost=CostfG(exper,spect)
        TRACK.record(wcost,lambda: np.array(points,float))
        return wcost
    TRACK.reset()
    Ham=deepcopy(Hamer)
    Exp=deepcopy(Expe)
    Var=deepcopy(Vara)
    initexper=exper
    lowfron=[]
    hifron=[]
    bplayer=0
    bcost=60
    if mode=='p':
        funtiona=Powder#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Eresonant#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    if datype=='data':
        exper=exper
    elif datype=='integral':
        fielda=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    else:
        raise ValueError(f"Data type {datype} is not supported. Use integral or data instead.")
    if Var.g!=0.0:
        lowfron.extend([Var.g[0],Var.g[2],Var.g[4]])
        hifron.extend([Var.g[1],Var.g[3],Var.g[5]])
    if Var.A!=0.0:
        lowfron.extend([Var.A[0],Var.A[2],Var.A[4]])
        hifron.extend([Var.A[1],Var.A[3],Var.A[5]])
    if Var.Q!=0.0:
        lowfron.extend([Var.Q[0],Var.Q[2],Var.Q[4]])
        hifron.extend([Var.Q[1],Var.Q[3],Var.Q[5]])
    if Var.D!=0.0:
        lowfron.extend([Var.D[0],Var.D[2]])
        hifron.extend([Var.D[1],Var.D[3]])
    if np.any(Var.Hpp):
        lowfron.extend([Var.Hpp[0],Var.Hpp[2]])
        hifron.extend([Var.Hpp[1],Var.Hpp[3]])
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    nparams=len(lowfron)
    poles=35*nparams
    if poles%2!=0:
        poles+=1
    population=np.zeros((poles,nparams))
    np.random.seed(int(seed))
    for ika in range(0,nparams):
        population[:,ika]=np.random.uniform(lowfron[ika],hifron[ika],poles)
    fitprice=np.zeros(poles)
    for jae in range(poles):
        fitprice[jae]=Fincost(population[jae],funcname)
    itea=0
    try:
        while itea<maximal:
            if stopvar.is_set():
                print("Process stopped by user.")
                stopvar.clear()
                break
            orden=np.argsort(fitprice)
            population=population[orden]
            fitprice=fitprice[orden]
            bcost=fitprice[0]
            bplayer=np.copy(population[0])
            if itea%5==0:
                _=Fincost(bplayer,funcname)
                print(f"Generation: {itea} | Best cost: {bcost:.5e}")
                print("="*50)
                if Var.g!=0.0:
                    print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
                if Var.A!=0.0:
                    print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
                if Var.Q!=0.0:
                    print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
                if Var.D!=0.0:
                    print(f'D={Ham.D[0]} | E={Ham.D[1]}')
                if np.any(Var.Hpp):
                    print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
            if bcost<eps:
                print("Data converged.")
                break
            pool=np.zeros((poles,nparams))
            for jae in range(poles):
                tchos=np.random.randint(0,poles,size=5)
                costt=fitprice[tchos]
                win=tchos[np.argmin(costt)]
                pool[jae]=population[win]
            npopulation=np.zeros((poles,nparams))
            npopulation[0]=bplayer
            for iq in range(1,poles,2):
                vat1=pool[iq]
                vat2=pool[iq+1] if (iq+1<poles) else pool[0]
                son1,son2=Cross(vat1,vat2)
                son1=Mutategauss(son1,lowfron,hifron)
                son2=Mutategauss(son2,lowfron,hifron)
                npopulation[iq]=son1
                if (iq+1<poles):
                    npopulation[iq+1]=son2
            population=npopulation
            for jae in range(poles):
                if jae==0:
                    fitprice[jae]=bcost
                else:
                    fitprice[jae]=Fincost(population[jae],funcname)
            
            itea+=1
    except KeyboardInterrupt:
        _=Fincost(bplayer,funcname)
        print("\n"+"="*50)
        print(f"Process stopped at generation: {itea}, with best cost: {bcost:.5e}")
        print("="*50)
        if Var.g!=0.0:
            print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
        if Var.A!=0.0:
            print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
        if Var.Q!=0.0:
            print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
        if Var.D!=0.0:
            print(f'D={Ham.D[0]} | E={Ham.D[1]}')
        if np.any(Var.Hpp):
            print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
        fitresult=Buildres(bplayer,Ham,Exp,initexper,Var,funcname,Mr,method='Genethic A',success=False,message='Genetic algorithm stopped by user',iterations=itea,dattype=datype)
        return fitresult.Ham,fitresult
    _=Fincost(bplayer,funcname)        
    print("\n"+"="*50)
    print(f"Process stopped at generation: {itea}, with best cost: {bcost:.5e}")
    print("="*50)
    if Var.g!=0.0:
        print(f'gx={Ham.g[0]} | gy={Ham.g[1]} | gz={Ham.g[2]}')
    if Var.A!=0.0:
        print(f'Ax={Ham.A[0]} | Ay={Ham.A[1]} | Az={Ham.A[2]}')
    if Var.Q!=0.0:
        print(f'Qx={Ham.Q[0]} | Qy={Ham.Q[1]} | Qz={Ham.Q[2]}')
    if Var.D!=0.0:
        print(f'D={Ham.D[0]} | E={Ham.D[1]}')
    if np.any(Var.Hpp):
        print(f'Hppg={Ham.Hpp[0]} | Hppl={Ham.Hpp[1]}')
    fitresult=Buildres(bplayer,Ham,Exp,initexper,Var,funcname,Mr,method='Genethic A',success=(itea<maximal),message='Converged with the genetic algorithm',iterations=itea,dattype=datype)
    return fitresult.Ham,fitresult

def Genio2(Hamer,Expe,Vara,exper,eps=1e-10,maximal=30,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting function for the experimental data using the genetic algorithm for multysystems.
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of generations to evalue.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    spce : np.array
        Best adjusted spectrum using the Mulpol or Music functions.
    '''

    def Fincost(points,funcname):
        if stopvar.is_set():
            return 1e6
        jl=0
        for i in range(len(Ham.Mulham)):
            if Var.Mvary[i].g!=0.0:
                gt=points[jl:jl+3]
                if (gt[0]<Var.Mvary[i].g[0] or gt[0]>Var.Mvary[i].g[1] or gt[1]<Var.Mvary[i].g[2] or gt[1]>Var.Mvary[i].g[3] or gt[2]<Var.Mvary[i].g[4] or gt[2]>Var.Mvary[i].g[5]):
                    return 1e6
                Ham.Mulham[i].g=gt
                jl+=3
            if Var.Mvary[i].A!=0.0:
                At=points[jl:jl+3]
                if (At[0]<Var.Mvary[i].A[0] or At[0]>Var.Mvary[i].A[1] or At[1]<Var.Mvary[i].A[2] or At[1]>Var.Mvary[i].A[3] or At[2]<Var.Mvary[i].A[4] or At[2]>Var.Mvary[i].A[5]):
                    return 1e6
                Ham.Mulham[i].A=At
                jl+=3
            if Var.Mvary[i].Q!=0.0:
                Qt=points[jl:jl+3]
                if (Qt[0]<Var.Mvary[i].Q[0] or Qt[0]>Var.Mvary[i].Q[1] or Qt[1]<Var.Mvary[i].Q[2] or Qt[1]>Var.Mvary[i].Q[3] or Qt[2]<Var.Mvary[i].Q[4] or Qt[2]>Var.Mvary[i].Q[5]):
                    return 1e6
                Ham.Mulham[i].Q=Qt
                jl+=3
            if Var.Mvary[i].D!=0.0:
                Dt=points[jl:jl+2]
                if (Dt[0]<Var.Mvary[i].D[0] or Dt[0]>Var.Mvary[i].D[1] or Dt[1]<Var.Mvary[i].D[2] or Dt[1]>Var.Mvary[i].D[3]):
                    return 1e6
                Ham.Mulham[i].D=Dt
                jl+=2
            if np.any(Var.Mvary[i].Hpp):
                Htp=points[jl:jl+2]
                if (Htp[0]<Var.Mvary[i].Hpp[0] or Htp[0]>Var.Mvary[i].Hpp[1] or Htp[1]<Var.Mvary[i].Hpp[2] or Htp[1]>Var.Mvary[i].Hpp[3]):
                    return 1e6
                Ham.Mulham[i].Hpp=Htp
                jl+=2
        try:
            if funcname in ['Mulpol']:
                fieldae,spect=funtiona(Ham,Exp,M=Mr,graph=False)
            elif funcname in ['Music']:
                fieldae,spect=funtiona(Ham,Exp,graph=False,table=False)
            if datype=='integral':
                spect=scii.cumulative_trapezoid(spect,fieldae,initial=0)
            wcost=CostfG(exper,spect)
            TRACK.record(wcost,lambda: np.array(points,float))
            return wcost
        except Exception:
            return 1e6
    TRACK.reset()
    Ham=deepcopy(Hamer)
    Exp=deepcopy(Expe)
    Var=deepcopy(Vara)
    initexper=exper
    lowfron=[]
    hifron=[]
    bplayer=0
    if mode=='p':
        funtiona=Mulpol#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Music#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    pointx=[]
    lowfron=[]
    hifron=[]
    if datype=='data':
        exper=exper
    elif datype=='integral':
        fielda=np.linspace(Exp.Frange[0],Exp.Frange[1],Exp.Points)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    else:
        raise ValueError(f"Data type {datype} is not supported. Use integral or data instead.")
    numberes=len(Ham.Mulham)
    for lke in range(0,numberes):
        if Var.Mvary[lke].g!=0.0:
            pointx.extend(Ham.Mulham[lke].g)
            lowfron.extend([Var.Mvary[lke].g[0],Var.Mvary[lke].g[2],Var.Mvary[lke].g[4]])
            hifron.extend([Var.Mvary[lke].g[1],Var.Mvary[lke].g[3],Var.Mvary[lke].g[5]])
        if Var.Mvary[lke].A!=0.0:
            pointx.extend(Ham.Mulham[lke].A)
            lowfron.extend([Var.Mvary[lke].A[0],Var.Mvary[lke].A[2],Var.Mvary[lke].A[4]])
            hifron.extend([Var.Mvary[lke].A[1],Var.Mvary[lke].A[3],Var.Mvary[lke].A[5]])
        if Var.Mvary[lke].Q!=0.0:
            pointx.extend(Ham.Mulham[lke].Q)
            lowfron.extend([Var.Mvary[lke].Q[0],Var.Mvary[lke].Q[2],Var.Mvary[lke].Q[4]])
            hifron.extend([Var.Mvary[lke].Q[1],Var.Mvary[lke].Q[3],Var.Mvary[lke].Q[5]])
        if Var.Mvary[lke].D!=0.0:
            pointx.extend(Ham.Mulham[lke].D)
            lowfron.extend([Var.Mvary[lke].D[0],Var.Mvary[lke].D[2]])
            hifron.extend([Var.Mvary[lke].D[1],Var.Mvary[lke].D[3]])
        if np.any(Var.Mvary[lke].Hpp):
            pointx.extend(Ham.Mulham[lke].Hpp)
            lowfron.extend([Var.Mvary[lke].Hpp[0],Var.Mvary[lke].Hpp[2]])
            hifron.extend([Var.Mvary[lke].Hpp[1],Var.Mvary[lke].Hpp[3]])
    pointx=np.array(pointx,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    nparams=len(lowfron)
    poles=35*nparams
    if poles%2!=0:
        poles+=1
    population=np.zeros((poles,nparams))
    np.random.seed(int(seed))
    for ika in range(0,nparams):
        population[:,ika]=np.random.uniform(lowfron[ika],hifron[ika],poles)
    fitprice=np.zeros(poles)
    for jae in range(poles):
        fitprice[jae]=Fincost(population[jae],funcname)
    itea=0
    try:
        while itea<maximal:
            if stopvar.is_set():
                print("Process stopped by user.")
                stopvar.clear()
                break
            orden=np.argsort(fitprice)
            population=population[orden]
            fitprice=fitprice[orden]
            bcost=fitprice[0]
            bplayer=np.copy(population[0])
            if itea%5==0:
                _=Fincost(bplayer,funcname)
                print(f"Generation: {itea} | Best cost: {bcost:.5e}")
                for i in range(len(Ham.Mulham)):
                    print(f"--- System {i+1} ---")
                    if Var.Mvary[i].g!=0.0:
                        print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
                    if Var.Mvary[i].A!=0.0:
                        print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
                    if Var.Mvary[i].Q!=0.0:
                        print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
                    if Var.Mvary[i].D!=0.0:
                        print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
                    if np.any(Var.Mvary[i].Hpp):
                        print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
            if bcost<eps:
                print("Data converged.")
                break
            pool=np.zeros((poles,nparams))
            for jae in range(poles):
                tchos=np.random.randint(0,poles,size=5)
                costt=fitprice[tchos]
                win=tchos[np.argmin(costt)]
                pool[jae]=population[win]
            npopulation=np.zeros((poles,nparams))
            npopulation[0]=bplayer
            for iq in range(1,poles,2):
                vat1=pool[iq]
                vat2=pool[iq+1] if (iq+1<poles) else pool[0]
                son1,son2=Cross(vat1,vat2)
                son1=Mutategauss(son1,lowfron,hifron)
                son2=Mutategauss(son2,lowfron,hifron)
                npopulation[iq]=son1
                if (iq+1<poles):
                    npopulation[iq+1]=son2
            population=npopulation
            for jae in range(poles):
                if jae==0:
                    fitprice[jae]=bcost
                else:
                    fitprice[jae]=Fincost(population[jae],funcname)
            itea+=1
    except KeyboardInterrupt:
        _=Fincost(bplayer,funcname)
        print("\n"+"="*50)
        print(f"Process stopped at generation: {itea}, with best cost: {bcost:.5e}")
        print("="*50)
        _=Fincost(bplayer,funcname)
        for i in range(len(Ham.Mulham)):
            print(f"--- System {i+1} ---")
            if Var.Mvary[i].g!=0.0:
                print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
            if Var.Mvary[i].A!=0.0:
                print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
            if Var.Mvary[i].Q!=0.0:
                print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
            if Var.Mvary[i].D!=0.0:
                print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
            if np.any(Var.Mvary[i].Hpp):
                print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
        fitresult=Buildres(bplayer,Ham,Exp,initexper,Var,funcname,Mr,method='Genethic A',success=False,message='Genetic algorithm stopped by user',iterations=itea,dattype=datype)
        return fitresult.Ham,fitresult
    print("\n"+"="*50)
    print(f"Process stopped at generation: {itea}, with best cost: {bcost:.5e}")
    print("="*50)
    _=Fincost(bplayer,funcname)
    for i in range(len(Ham.Mulham)):
        print(f"--- System {i+1} ---")
        if Var.Mvary[i].g!=0.0:
            print(f'gx={Ham.Mulham[i].g[0]:.4f} | gy={Ham.Mulham[i].g[1]:.4f} | gz={Ham.Mulham[i].g[2]:.4f}')
        if Var.Mvary[i].A!=0.0:
            print(f'Ax={Ham.Mulham[i].A[0]:.4f} | Ay={Ham.Mulham[i].A[1]:.4f} | Az={Ham.Mulham[i].A[2]:.4f}')
        if Var.Mvary[i].Q!=0.0:
            print(f'Qx={Ham.Mulham[i].Q[0]:.4f} | Qy={Ham.Mulham[i].Q[1]:.4f} | Qz={Ham.Mulham[i].Q[2]:.4f}')
        if Var.Mvary[i].D!=0.0:
            print(f'D={Ham.Mulham[i].D[0]:.4f} | E={Ham.Mulham[i].D[1]:.4f}')
        if np.any(Var.Mvary[i].Hpp):
            print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
    fitresult=Buildres(bplayer,Ham,Exp,initexper,Var,funcname,Mr,method='Genethic A',success=(itea<maximal),message='Converged with the genetic algorithm',iterations=itea,dattype=datype)
    return fitresult.Ham,fitresult

def Genio(Hamer,Expe,Vara,exper,eps=1e-10,maximal=100,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting function for the experimental data using the genetic algorithm. Creates a population of 35*N individuals, where N is the number of parameters that will change, that have combinations of the possible values of the parameters. This individuals are evaluated, select and cross to produce a new population. 
    
    The algorithm uses a normal distribution for the mutation probability and a the numpy random int generator for the cross probability. 
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    eps : float
        Tolerance value for the error. Default is 1e-10
    maximal : int
        Max. number of generations to evalue.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution
    Mr : int
        Number of divisions for the Delaunay grid.
     
    Returns
    -------
    Ham : Class
        Container for the fitted hamiltonian parameters.
    fitresult : Class 
        Container for the results of the Fitting function.
    '''
    if type(Hamer)==Multham:
         if Expe.Mexp[0].Points!=len(exper):
            Expe.Mexp[0].Points=len(exper)
         return Genio2(Hamer,Expe,Vara,exper,eps,maximal,datype,mode,seed,Mr)
    elif type(Hamer)==Hval:
         if Expe.Points!=len(exper):
            Expe.Points=len(exper)
         return Genio1(Hamer,Expe,Vara,exper,eps,maximal,datype,mode,seed,Mr)

def Costf(exper,intens,metric='L2'):
    if metric=='rmse':
        # Root Mean Squared Error
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        error=np.sqrt(np.mean((intens-exper)**2))
        return error
    elif metric=='correlation':
        norm=np.sum(intens*intens)
        if norm==0 or np.max(intens)<1e-10:
            return 1e6
        maxe=np.max(np.abs(exper))
        maxi=np.max(np.abs(intens))
        exper=exper/maxe
        intens=intens/maxi
        corr=np.corrcoef(exper,intens)
        pearson=corr[0, 1]
        if np.isnan(pearson):
            return 1e6
        return 1-pearson
    elif metric=='L2':
        normi=np.linalg.norm(intens)
        if normi==0 or np.isnan(normi):
            return 1e6
        norme=np.linalg.norm(exper)
        enorm=exper/norme
        inorm=intens/normi
        return np.sum((inorm-enorm)**2)

def Metrostair(Hamer,Exp,Var,date,stepsize,ocos,para,variable,funcname,iwas,jwas,kwas,weight,hulk,datype='data'):    
    Ham=deepcopy(Hamer)
    if 'g' in variable:
        Ham.g[0]+=np.random.normal(0,stepsize['gx'])
        Ham.g[1]+=np.random.normal(0,stepsize['gy'])
        Ham.g[2]+=np.random.normal(0,stepsize['gz'])
        if not (Var.g[0]<=Ham.g[0]<=Var.g[1] and Var.g[2]<=Ham.g[1]<=Var.g[3] and Var.g[4]<=Ham.g[2]<=Var.g[5]):
            return Hamer,ocos,False

    if 'A' in variable:
        Ham.A[0]+=np.random.normal(0,stepsize['Ax'])
        Ham.A[1]+=np.random.normal(0,stepsize['Ay'])
        Ham.A[2]+=np.random.normal(0,stepsize['Az'])
        if not (Var.A[0]<=Ham.A[0]<=Var.A[1] and Var.A[2]<=Ham.A[1]<=Var.A[3] and Var.A[4]<=Ham.A[2]<=Var.A[5]):
            return Hamer,ocos,False

    if 'Q' in variable:
        Ham.Q[0]+=np.random.normal(0,stepsize['Qx'])
        Ham.Q[1]+=np.random.normal(0,stepsize['Qy'])
        Ham.Q[2]+=np.random.normal(0,stepsize['Qz'])
        if not (Var.Q[0]<=Ham.Q[0]<=Var.Q[1] and Var.Q[2]<=Ham.Q[1]<=Var.Q[3] and Var.Q[4]<=Ham.Q[2]<=Var.Q[5]):
            return Hamer,ocos,False

    if 'D' in variable:
        Ham.D[0]+=np.random.normal(0,stepsize['D'])
        Ham.D[1]+=np.random.normal(0,stepsize['E'])
        if not (Var.D[0]<=Ham.D[0]<=Var.D[1] and Var.D[2]<=Ham.D[1]<=Var.D[3]):
            return Hamer,ocos,False

    if 'Hpp' in variable:
        Ham.Hpp[0]+=np.random.normal(0,stepsize['Hpp1'])
        Ham.Hpp[1]+=np.random.normal(0,stepsize['Hpp2'])
        if not (Var.Hpp[0]<=Ham.Hpp[0]<=Var.Hpp[1] and Var.Hpp[2]<=Ham.Hpp[1]<=Var.Hpp[3]):
            return Hamer,ocos,False
    if funcname in ['Calpowder']:
        fielda,intena=Calpowder(Ham,Exp,iwas,jwas,kwas,weight,hulk)
    elif funcname in ['Eresonant']:
        fielda,intena=Eresonant(Ham,Exp,graph=False,table=False)
    if datype=='data':
        ncos=Costf(date,intena)
        TRACK.record(ncos, lambda: Packtoscipy(Ham,Var)[0])
    elif datype=='integral':
        intenat=scii.cumulative_trapezoid(intena,fielda,initial=0)
        ncos=Costf(date,intenat)
        TRACK.record(ncos,lambda: Packtoscipy(Ham,Var)[0])
    else:
        ncos=1e6

    if np.isnan(ncos) or ncos>=1e6:
        return Hamer,ocos,False
    if ncos<ocos:
        return Ham,ncos,True
    else:
        Tsaf=max(para,1e-8)
        prob=np.exp(-(np.sqrt(ncos)-np.sqrt(ocos))/Tsaf)
        if np.random.rand()<prob:
            return Ham,ncos,True
        else:
            return Hamer,ocos,False

def Metro1(Hamer,Exp,Var,dat,maximal,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting adjutsment of the experimental data using a modified Metrópolis-Simulated annealing approach for a single system.
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Exp : Class
        Container for the experimental conditions.
    Var : Class
        Container for the range and parameters to vary.
    dat : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    spc : np.array
        Best adjusted spectrum using the Powder or Eresonant functions.
    '''

    np.random.seed(int(seed))
    TRACK.reset()
    Ham1=deepcopy(Hamer)
    initdat=dat
    iwas,jwas,kwas,weight,hulk=Delaunay(Exp,Mr)
    if mode=='p':
        funtiona=Calpowder#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Eresonant#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    #First iteration
    if funcname in ['Calpowder']:
        fielda,intena=funtiona(Ham1,Exp,iwas,jwas,kwas,weight,hulk)
    elif funcname in ['Eresonant']:
        fielda,intena=funtiona(Ham1,Exp,graph=False,table=False)
    if datype=='data':
        ct1=Costf(dat,intena)
        TRACK.record(ct1,lambda: Packtoscipy(Ham1,Var)[0])
    if datype=='integral':
        dat=scii.cumulative_trapezoid(dat,fielda,initial=0)
        intena=scii.cumulative_trapezoid(intena,fielda,initial=0)
        ct1=Costf(dat,intena)
        TRACK.record(ct1,lambda: Packtoscipy(Ham1,Var)[0])
    pointx=[]
    stp={}
    lowfron=[]
    hifron=[]
    pg,pa,pq,pdr,php=20,50,50,50,50
    if Var.g!=0.0:
        pointx.extend(Ham1.g)
        stp['gx']=(Var.g[1]-Var.g[0])/pg
        stp['gy']=(Var.g[3]-Var.g[2])/pg
        stp['gz']=(Var.g[5]-Var.g[4])/pg
        lowfron.extend([Var.g[0],Var.g[2],Var.g[4]])
        hifron.extend([Var.g[1],Var.g[3],Var.g[5]])
    if Var.A!=0.0:
        pointx.extend(Ham1.A)
        stp['Ax']=(Var.A[1]-Var.A[0])/pa
        stp['Ay']=(Var.A[3]-Var.A[2])/pa
        stp['Az']=(Var.A[5]-Var.A[4])/pa
        lowfron.extend([Var.A[0],Var.A[2],Var.A[4]])
        hifron.extend([Var.A[1],Var.A[3],Var.A[5]])
    if Var.Q!=0.0:
        pointx.extend(Ham1.Q)
        stp['Qx']=(Var.Q[1]-Var.Q[0])/pq
        stp['Qy']=(Var.Q[3]-Var.Q[2])/pq
        stp['Qz']=(Var.Q[5]-Var.Q[4])/pq
        lowfron.extend([Var.Q[0],Var.Q[2],Var.Q[4]])
        hifron.extend([Var.Q[1],Var.Q[3],Var.Q[5]])
    if Var.D!=0.0:
        pointx.extend(Ham1.D)
        stp['D']=(Var.D[1]-Var.D[0])/pdr
        stp['E']=(Var.D[3]-Var.D[2])/pdr
        lowfron.extend([Var.D[0],Var.D[2]])
        hifron.extend([Var.D[1],Var.D[3]])
    if np.any(Var.Hpp):
        pointx.extend(Ham1.Hpp)
        stp['Hpp1']=(Var.Hpp[1]-Var.Hpp[0])/php
        stp['Hpp2']=(Var.Hpp[3]-Var.Hpp[2])/php
        lowfron.extend([Var.Hpp[0],Var.Hpp[2]])
        hifron.extend([Var.Hpp[1],Var.Hpp[3]])

    pointx=np.array(pointx,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    securestp=stp.copy()
    metropa=np.sqrt(ct1)*0.008 #Montecarlo Temperature
    metropas=metropa
    besterror=ct1
    laster=ct1
    defcon=0
    bestHam=deepcopy(Ham1)
    try:
        for gama in range(0,maximal):
            if stopvar.is_set():
                stopvar.clear()
                break
            acepv=0
            tries=0
            if Var.g!=0.0:
                hemetro=metropa
                opa=0
                while opa<pg:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,['g'],funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                Ham1=deepcopy(bestHam)
                ct1=besterror
            if Var.A!=0.0:
                hemetro=metropa
                opa=0
                while opa<pa:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,['A'],funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                Ham1=deepcopy(bestHam)
                ct1=besterror
            if Var.Q!=0.0:
                hemetro=metropa
                opa=0
                while opa<pq:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,['Q'],funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                Ham1=deepcopy(bestHam)
                ct1=besterror
            if Var.D!=0.0:
                hemetro=metropa
                opa=0
                while opa<pdr:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,['D'],funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                Ham1=deepcopy(bestHam)
                ct1=besterror
            if np.any(Var.Hpp):
                hemetro=metropa
                opa=0
                while opa<php:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,['Hpp'],funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                Ham1=deepcopy(bestHam)
                ct1=besterror

            #Last try with all
            varact=[]
            if Var.g!=0.0: varact.append('g')
            if Var.A!=0.0: varact.append('A')
            if Var.D!=0.0: varact.append('D')
            if Var.Q!=0.0: varact.append('Q')
            if np.any(Var.Hpp): varact.append('Hpp')
            if len(varact)>1:
                hemetro=metropa*1.5
                opa=0
                while opa<30:
                    Ham1,ct1,acep=Metrostair(Ham1,Exp,Var,dat,stp,ct1,hemetro,varact,funcname,iwas,jwas,kwas,weight,hulk,datype)
                    if acep:
                        acepv+=1
                    tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)

            metropa=metropa*0.95
            arate=(acepv/tries)*100 if tries> 0 else 0
            if arate>40.0:
                for k in stp.keys():
                    stp[k]*=1.05
            elif arate<20.0:
                for k in stp.keys():
                    stp[k]*=0.95
            #Shake values
            improv=laster-besterror
            if improv<1e-3*laster:
                defcon+=1
            else:
                defcon=0
                laster=besterror
            if defcon>=10:
                print(f"|--- Varying values to avoid stagnation ---|")
                metropa=metropas*5.0
                for k in stp.keys():
                    stp[k]*=3.0
                Ham1=deepcopy(bestHam)
                defcon=0

            #Awake the process
            if arate<10.0 and gama>20:
                metropa=metropa*100.0
                if metropa>metropas:
                    metropa=metropas
                    for k in stp.keys():
                        stp[k]=securestp[k]/2
                for k in stp.keys():
                    stp[k]*=2.0
                Ham1=deepcopy(bestHam)
                print(f"|---Process heated up---|")
            if gama%10==0:
                print(f"Step {gama} | Error: {ct1:.5e} | Best one: {besterror:.5e} | Rate: {arate:.1f}% | T: {metropa:.4e}")
                if Var.g!=0.0:
                    print(f'gx={bestHam.g[0]} | gy={bestHam.g[1]} | gz={bestHam.g[2]}')
                if Var.A!=0.0:
                    print(f'Ax={bestHam.A[0]} | Ay={bestHam.A[1]} | Az={bestHam.A[2]}')
                if Var.Q!=0.0:
                    print(f'Qx={bestHam.Q[0]} | Qy={bestHam.Q[1]} | Qz={bestHam.Q[2]}')
                if Var.D!=0.0:
                    print(f'D={bestHam.D[0]},E={bestHam.D[1]}')
                if np.any(Var.Hpp):
                    print(f'Hppg={bestHam.Hpp[0]},Hppl={bestHam.Hpp[1]}')
    except KeyboardInterrupt:
        bestpoint,_,_=Packtoscipy(bestHam,Var)
        print("\n"+"="*50)
        print(f"Process stopped at iteration: {gama}, with best cost {besterror:.5e}")
        print("="*50)
        if Var.g!=0.0:
            print(f'gx={bestHam.g[0]} | gy={bestHam.g[1]} | gz={bestHam.g[2]}')
        if Var.A!=0.0:
            print(f'Ax={bestHam.A[0]} | Ay={bestHam.A[1]} | Az={bestHam.A[2]}')
        if Var.Q!=0.0:
            print(f'Qx={bestHam.Q[0]} | Qy={bestHam.Q[1]} | Qz={bestHam.Q[2]}')
        if Var.D!=0.0:
            print(f'D={bestHam.D[0]},E={bestHam.D[1]}')
        if np.any(Var.Hpp):
            print(f'Hppg={bestHam.Hpp[0]},Hppl={bestHam.Hpp[1]}')
        fitresult=Buildres(bestpoint,Ham1,Exp,initdat,Var,funcname,Mr,method='Metropolis',success=False,message='Metropolis stopped by user',iterations=gama,dattype=datype)
        return fitresult.Ham, fitresult
    bestpoint,_,_=Packtoscipy(bestHam,Var)
    print("\n"+"="*50)
    print(f"Process stopped at iteration: {gama}, with best cost {besterror:.5e}")
    print("="*50)
    if Var.g!=0.0:
        print(f'gx={bestHam.g[0]} | gy={bestHam.g[1]} | gz={bestHam.g[2]}')
    if Var.A!=0.0:
        print(f'Ax={bestHam.A[0]} | Ay={bestHam.A[1]} | Az={bestHam.A[2]}')
    if Var.Q!=0.0:
        print(f'Qx={bestHam.Q[0]} | Qy={bestHam.Q[1]} | Qz={bestHam.Q[2]}')
    if Var.D!=0.0:
        print(f'D={bestHam.D[0]} | E={bestHam.D[1]}')
    if np.any(Var.Hpp):
        print(f'Hppg={bestHam.Hpp[0]} | Hppl={bestHam.Hpp[1]}')
    fitresult=Buildres(bestpoint,Ham1,Exp,initdat,Var,funcname,Mr,method='Metropolis',success=True,message='Converged with Metropolis',iterations=gama,dattype=datype)
    return fitresult.Ham, fitresult

def Metrostair2(Hamer,Exp,Var,date,stepsize,ocos,para,variable,aktsys,funcname,datype='data',Mr=70):
    Ham=deepcopy(Hamer)
    vma=Var.Mvary[aktsys]
    hma=Ham.Mulham[aktsys]
    if 'g' in variable:
        hma.g[0]+=np.random.normal(0,stepsize[f'gx_{aktsys}'])
        hma.g[1]+=np.random.normal(0,stepsize[f'gy_{aktsys}'])
        hma.g[2]+=np.random.normal(0,stepsize[f'gz_{aktsys}'])
        if not (vma.g[0]<=hma.g[0]<=vma.g[1] and vma.g[2]<=hma.g[1]<=vma.g[3] and vma.g[4]<=hma.g[2]<=vma.g[5]):
            return Hamer,ocos,False

    if 'A' in variable:
        hma.A[0]+=np.random.normal(0,stepsize[f'Ax_{aktsys}'])
        hma.A[1]+=np.random.normal(0,stepsize[f'Ay_{aktsys}'])
        hma.A[2]+=np.random.normal(0,stepsize[f'Az_{aktsys}'])
        if not (vma.A[0]<=hma.A[0]<=vma.A[1] and vma.A[2]<=hma.A[1]<=vma.A[3] and vma.A[4]<=hma.A[2]<=vma.A[5]):
            return Hamer,ocos,False

    if 'Q' in variable:
        hma.Q[0]+=np.random.normal(0,stepsize[f'Qx_{aktsys}'])
        hma.Q[1]+=np.random.normal(0,stepsize[f'Qy_{aktsys}'])
        hma.Q[2]+=np.random.normal(0,stepsize[f'Qz_{aktsys}'])
        if not (vma.Q[0]<=hma.Q[0]<=vma.Q[1] and vma.Q[2]<=hma.Q[1]<=vma.Q[3] and vma.Q[4]<=hma.Q[2]<=vma.Q[5]):
            return Hamer,ocos,False

    if 'D' in variable:
        hma.D[0]+=np.random.normal(0,stepsize[f'D_{aktsys}'])
        hma.D[1]+=np.random.normal(0,stepsize[f'E_{aktsys}'])
        if not (vma.D[0]<=hma.D[0]<=vma.D[1] and vma.D[2]<=hma.D[1]<=vma.D[3]):
            return Hamer,ocos,False

    if 'Hpp' in variable:
        hma.Hpp[0]+=np.random.normal(0,stepsize[f'Hpp1_{aktsys}'])
        hma.Hpp[1]+=np.random.normal(0,stepsize[f'Hpp2_{aktsys}'])
        if not (vma.Hpp[0]<=hma.Hpp[0]<=vma.Hpp[1] and vma.Hpp[2]<=hma.Hpp[1]<=vma.Hpp[3]):
            return Hamer,ocos,False
    if funcname in ['Mulpol']:
        fielda,intena=Mulpol(Ham,Exp,M=Mr,graph=False)
    elif funcname in ['Music']:
        fielda,intena=Music(Ham,Exp,graph=False,table=False)
    if datype=='data':
        ncos=Costf(date,intena)
        TRACK.record(ncos,lambda: Packtoscipy(Ham,Var)[0])
    elif datype=='integral':
        intenat=scii.cumulative_trapezoid(intena,fielda,initial=0)
        ncos=Costf(date,intenat)
        TRACK.record(ncos,lambda: Packtoscipy(Ham,Var)[0])
    else:
        ncos=1e6

    if np.isnan(ncos) or ncos>=1e6:
        return Hamer,ocos,False
    if ncos<ocos:
        return Ham,ncos,True
    else:
        Tsaf=max(para,1e-8)
        prob=np.exp(-(np.sqrt(ncos)-np.sqrt(ocos))/Tsaf)
        if np.random.rand()<prob:
            return Ham,ncos,True
        else:
            return Hamer,ocos,False

def Metro2(Hamer,Exp,Var,dat,maximal,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting adjutsment of the experimental data using a modified Metrópolis-Simulated annealing approach for multiple systems.
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Exp : Class
        Container for the experimental conditions.
    Var : Class
        Container for the range and parameters to vary.
    dat : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    spc : np.array
        Best adjusted spectrum using the Mulpol or Music functions.
    '''
    Ham1=deepcopy(Hamer)
    TRACK.reset()
    initdat=dat        
    if mode=='p':
        funtiona=Mulpol#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Music#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    #First iteration
    if funcname in ['Mulpol']:
        fielda,intena=Mulpol(Ham1,Exp,M=Mr,graph=False)
    elif funcname in ['Music']:
        fielda,intena=Music(Ham1,Exp,graph=False,table=False)
    if datype=='data':
        ct1=Costf(dat,intena)
        TRACK.record(ct1,lambda: Packtoscipy(Ham1,Var)[0])
    if datype=='integral':
        dat=scii.cumulative_trapezoid(dat,fielda,initial=0)
        intena=scii.cumulative_trapezoid(intena,fielda,initial=0)
        ct1=Costf(dat,intena)
        TRACK.record(ct1,lambda: Packtoscipy(Ham1,Var)[0])
    pointx=[]
    stp={}
    lowfron=[]
    hifron=[]
    pg,pa,pq,pdr,php=20,50,50,50,50
    for ier in range(len(Ham1.Mulham)):
        vma=Var.Mvary[ier]
        hma=Ham1.Mulham[ier]
        if isinstance(vma.g,(list,np.ndarray)) or vma.g!=0.0:
            stp[f'gx_{ier}']=(vma.g[1]-vma.g[0])/pg
            stp[f'gy_{ier}']=(vma.g[3]-vma.g[2])/pg
            stp[f'gz_{ier}']=(vma.g[5]-vma.g[4])/pg
        if isinstance(vma.A,(list,np.ndarray)) or vma.A!=0.0:
            stp[f'Ax_{ier}']=(vma.A[1]-vma.A[0])/pa
            stp[f'Ay_{ier}']=(vma.A[3]-vma.A[2])/pa
            stp[f'Az_{ier}']=(vma.A[5]-vma.A[4])/pa
        if isinstance(vma.Q,(list, np.ndarray)) or vma.Q!=0.0:
            stp[f'Qx_{ier}']=(vma.Q[1]-vma.Q[0])/pq
            stp[f'Qy_{ier}']=(vma.Q[3]-vma.Q[2])/pq
            stp[f'Qz_{ier}']=(vma.Q[5]-vma.Q[4])/pq
        if isinstance(vma.D,(list,np.ndarray)) or vma.D!=0.0:
            stp[f'D_{ier}']=(vma.D[1]-vma.D[0])/pdr
            stp[f'E_{ier}']=(vma.D[3]-vma.D[2])/pdr
        if np.any(vma.Hpp):
            stp[f'Hpp1_{ier}']=(vma.Hpp[1]-vma.Hpp[0])/php
            stp[f'Hpp2_{ier}']=(vma.Hpp[3]-vma.Hpp[2])/php
    pointx=np.array(pointx,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    securestp=stp.copy()
    metropa=np.sqrt(ct1)*0.008 #Montecarlo Temperature
    metropas=metropa
    besterror=ct1
    laster=ct1
    defcon=0
    bestHam=deepcopy(Ham1)
    try:
        for gama in range(0,maximal):
            if stopvar.is_set():
                stopvar.clear()
                break
            acepv=0
            tries=0
            for ira in range(len(Ham1.Mulham)):
                if Var.Mvary[ira].g!=0.0:
                    hemetro=metropa
                    opa=0
                    while opa<pg:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,['g'],ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                    Ham1=deepcopy(bestHam)
                    ct1=besterror
                if Var.Mvary[ira].A!=0.0:
                    hemetro=metropa
                    opa=0
                    while opa<pa:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,['A'],ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                    hemetro=hemetro*0.99
                    opa+=1
                    if ct1<besterror:
                        besterror=ct1
                        bestHam=deepcopy(Ham1)
                    Ham1=deepcopy(bestHam)
                    ct1=besterror
                if Var.Mvary[ira].Q!=0.0:
                    hemetro=metropa
                    opa=0
                    while opa<pq:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,['Q'],ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                        hemetro=hemetro*0.99
                        opa+=1
                        if ct1<besterror:
                            besterror=ct1
                            bestHam=deepcopy(Ham1)
                    Ham1=deepcopy(bestHam)
                    ct1=besterror
                if Var.Mvary[ira].D!=0.0:
                    hemetro=metropa
                    opa=0
                    while opa<pdr:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,['D'],ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                        hemetro=hemetro*0.99
                        opa+=1
                        if ct1<besterror:
                            besterror=ct1
                            bestHam=deepcopy(Ham1)
                    Ham1=deepcopy(bestHam)
                    ct1=besterror
                if np.any(Var.Mvary[ira].Hpp):
                    hemetro=metropa
                    opa=0
                    while opa<php:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,['Hpp'],ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                        hemetro=hemetro*0.99
                        opa+=1
                        if ct1<besterror:
                            besterror=ct1
                            bestHam=deepcopy(Ham1)
                    Ham1=deepcopy(bestHam)
                    ct1=besterror

            #Last try with all
            hemetro=metropa*1.5
            opa=0
            while opa<30:
                for ira in range(len(Ham1.Mulham)):
                    varact=[]
                    if Var.Mvary[ira].g!=0.0:
                        varact.append('g')
                    if Var.Mvary[ira].A!=0.0:
                        varact.append('A')
                    if Var.Mvary[ira].D!=0.0:
                        varact.append('D')
                    if Var.Mvary[ira].Q!=0.0:
                        varact.append('Q') 
                    if np.any(Var.Mvary[ira].Hpp):
                        varact.append('Hpp')
                    if len(varact)>0:
                        Ham1,ct1,acep=Metrostair2(Ham1,Exp,Var,dat,stp,ct1,hemetro,varact,ira,funcname,datype,Mr)
                        if acep:
                            acepv+=1
                        tries+=1
                hemetro=hemetro*0.99
                opa+=1
                if ct1<besterror:
                    besterror=ct1
                    bestHam=deepcopy(Ham1)

            metropa=metropa*0.95
            arate=(acepv/tries)*100 if tries> 0 else 0
            if arate>40.0:
                for k in stp.keys():
                    stp[k]*=1.05
            elif arate<20.0:
                for k in stp.keys():
                    stp[k]*=0.95
            #Shake values
            improv=laster-besterror
            if improv<1e-3*laster:
                defcon+=1
            else:
                defcon=0
                laster=besterror
            if defcon>=10:
                print(f"|--- Varying values to avoid stagnation ---|")
                metropa=metropas*5.0
                for k in stp.keys():
                    stp[k]*=3.0
                Ham1=deepcopy(bestHam)
                defcon=0

            #Awake the process
            if arate<10.0 and gama>20:
                metropa=metropa*100.0
                if metropa>metropas:
                    metropa=metropas
                    for k in stp.keys():
                        stp[k]=securestp[k]/2
                for k in stp.keys():
                    stp[k]*=2.0
                Ham1=deepcopy(bestHam)
                print(f"|---Process heated up---|")
            if gama%10==0:
                print(f"Step {gama} | Error: {ct1:.5e} | Best one: {besterror:.5e} | Rate: {arate:.1f}% | T: {metropa:.4e}")
                for i in range(len(Ham1.Mulham)):
                    print(f"--- System {i+1} ---")
                    if Var.Mvary[i].g!=0.0:
                        print(f'gx={bestHam.Mulham[i].g[0]:.4f} | gy={bestHam.Mulham[i].g[1]:.4f} | gz={bestHam.Mulham[i].g[2]:.4f}')
                    if Var.Mvary[i].A!=0.0:
                        print(f'Ax={bestHam.Mulham[i].A[0]:.4f} | Ay={bestHam.Mulham[i].A[1]:.4f} | Az={bestHam.Mulham[i].A[2]:.4f}')
                    if Var.Mvary[i].Q!=0.0:
                        print(f'Qx={bestHam.Mulham[i].Q[0]:.4f} | Qy={bestHam.Mulham[i].Q[1]:.4f} | Qz={bestHam.Mulham[i].Q[2]:.4f}')
                    if Var.Mvary[i].D!=0.0:
                        print(f'D={bestHam.Mulham[i].D[0]:.4f} | E={bestHam.Mulham[i].D[1]:.4f}')
                    if np.any(Var.Mvary[i].Hpp):
                        print(f'Hppg={Ham.Mulham[i].Hpp[0]:.4f} | Hppl={Ham.Mulham[i].Hpp[1]:.4f}')
    except KeyboardInterrupt:
        bestpoint,_,_=Packtoscipy2(bestHam,Var)
        print("\n"+"="*50)
        print(f"Process stopped at iteration: {gama}, with best cost {besterror:.5e}")
        print("="*50)
        for i in range(len(Ham1.Mulham)):
            print(f"--- System {i+1} ---")
            if Var.Mvary[i].g!=0.0:
                print(f'gx={bestHam.Mulham[i].g[0]:.4f} | gy={bestHam.Mulham[i].g[1]:.4f} | gz={bestHam.Mulham[i].g[2]:.4f}')
            if Var.Mvary[i].A!=0.0:
                print(f'Ax={bestHam.Mulham[i].A[0]:.4f} | Ay={bestHam.Mulham[i].A[1]:.4f} | Az={bestHam.Mulham[i].A[2]:.4f}')
            if Var.Mvary[i].Q!=0.0:
                print(f'Qx={bestHam.Mulham[i].Q[0]:.4f} | Qy={bestHam.Mulham[i].Q[1]:.4f} | Qz={bestHam.Mulham[i].Q[2]:.4f}')
            if Var.Mvary[i].D!=0.0:
                print(f'D={bestHam.Mulham[i].D[0]:.4f} | E={bestHam.Mulham[i].D[1]:.4f}')
            if np.any(Var.Mvary[i].Hpp):
                print(f'Hppg={bestHam.Mulham[i].Hpp[0]:.4f} | Hppl={bestHam.Mulham[i].Hpp[1]:.4f}')
        fitresult=Buildres(bestpoint,Ham1,Exp,initdat,Var,funcname,Mr,method='Metropolis',success=False,message='Metropolis stopped by user',iterations=gama,dattype=datype)
        return fitresult.Ham, fitresult
    bestpoint,_,_=Packtoscipy2(bestHam,Var)
    print("\n"+"="*50)
    print(f"Process stopped at iteration: {gama}, with best cost {besterror:.5e}")
    print("="*50)
    for i in range(len(Ham1.Mulham)):
        print(f"--- System {i+1} ---")
        if Var.Mvary[i].g!=0.0:
            print(f'gx={bestHam.Mulham[i].g[0]:.4f} | gy={bestHam.Mulham[i].g[1]:.4f} | gz={bestHam.Mulham[i].g[2]:.4f}')
        if Var.Mvary[i].A!=0.0:
            print(f'Ax={bestHam.Mulham[i].A[0]:.4f} | Ay={bestHam.Mulham[i].A[1]:.4f} | Az={bestHam.Mulham[i].A[2]:.4f}')
        if Var.Mvary[i].Q!=0.0:
            print(f'Qx={bestHam.Mulham[i].Q[0]:.4f} | Qy={bestHam.Mulham[i].Q[1]:.4f} | Qz={bestHam.Mulham[i].Q[2]:.4f}')
        if Var.Mvary[i].D!=0.0:
            print(f'D={bestHam.Mulham[i].D[0]:.4f} | E={bestHam.Mulham[i].D[1]:.4f}')
        if np.any(Var.Mvary[i].Hpp):
            print(f'Hppg={bestHam.Mulham[i].Hpp[0]:.4f} | Hppl={bestHam.Mulham[i].Hpp[1]:.4f}')
    fitresult=Buildres(bestpoint,Ham1,Exp,initdat,Var,funcname,Mr,method='Metropolis',success=True,message='Converged with Metropolis',iterations=gama,dattype=datype)
    return fitresult.Ham, fitresult

def Metro(Hamer,Exp,Var,exper,maximal=2000,datype='data',mode='p',seed=451,Mr=70):
    '''
    Fitting adjutsment of the experimental data using a modified Metrópolis-Simulated annealing approach. First, the parameters in the Vary container are evaluated in range, one by one, using a gaussian distribution and then, all are varied at the same time. With every evaluation, the difference between the simulation and the data is calculated as a cost.
    
    Depending in the acceptance rate of the process, the temperature of the process change. If the acceptance is too low, the process heats up (the range of the evaluating values increases), if the accpetance is high, the process colds down (the range reduces).
    
    Parameters
    ----------
    
    Hamer: Class
        Container for the hamiltonian parameters.
    Exp : Class
        Container for the experimental conditions.
    Var : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    datype : str
        Type of data to do the fitting, can be 'data', the data as it's or 'integral', its first integral.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    seed : int
        Seed for the uniform distribution.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    Ham : Class
        Container for the fitted hamiltonian parameters.
    fitresult : Class 
        Container for the results of the Fitting function.
    '''
    if type(Hamer)==Multham:
         if Exp.Mexp[0].Points!=len(exper):
            Exp.Mexp[0].Points=len(exper)
         return Metro2(Hamer,Exp,Var,exper,maximal,datype,mode,seed,Mr)
    elif type(Hamer)==Hval:
         if Exp.Points!=len(exper):
            Exp.Points=len(exper)
         return Metro1(Hamer,Exp,Var,exper,maximal,datype,mode,seed,Mr)

def Packtoscipy(Ham,Var):
    pointx=[]
    lowfron=[]
    hifron=[]
    if Var.g!=0.0:
        pointx.extend(Ham.g)
        lowfron.extend([Var.g[0],Var.g[2],Var.g[4]])
        hifron.extend([Var.g[1],Var.g[3],Var.g[5]])
    if Var.A!=0.0:
        pointx.extend(Ham.A)
        lowfron.extend([Var.A[0],Var.A[2],Var.A[4]])
        hifron.extend([Var.A[1],Var.A[3],Var.A[5]])
    if Var.Q!=0.0:
        pointx.extend(Ham.Q)
        lowfron.extend([Var.Q[0],Var.Q[2],Var.Q[4]])
        hifron.extend([Var.Q[1],Var.Q[3],Var.Q[5]])
    if Var.D!=0.0:
        pointx.extend(Ham.D)
        lowfron.extend([Var.D[0],Var.D[2]])
        hifron.extend([Var.D[1],Var.D[3]])
    if np.any(Var.Hpp):
        pointx.extend(Ham.Hpp)
        lowfron.extend([Var.Hpp[0],Var.Hpp[2]])
        if Var.Hpp[0]==Var.Hpp[1]:
            hifron.extend([Var.Hpp[1]+0.0001,Var.Hpp[3]])
        elif Var.Hpp[2]==Var.Hpp[3]:
            hifron.extend([Var.Hpp[1],Var.Hpp[3]+0.0001])
        else:
            hifron.extend([Var.Hpp[1],Var.Hpp[3]])
    pointx=np.array(pointx,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    return pointx,lowfron,hifron

def UnpackHam(point,Hamer,Var):
    Ham=deepcopy(Hamer)
    idx=0
    if Var.g!=0.0:
        Ham.g=point[idx:idx+3]
        idx+=3
    if Var.A!=0.0:
        Ham.A=point[idx:idx+3]
        idx+=3
    if Var.Q!=0.0:
        Ham.Q=point[idx:idx+3]
        idx+=3
    if Var.D!=0.0:
        Ham.D=point[idx:idx+2]
        idx+=2
    if np.any(Var.Hpp):
        Ham.Hpp=point[idx:idx+2]
    return Ham
    
class Stopall(Exception):
    pass

def Residuals(point,Hame,Exp,exper,Vary,fname,Mr=70,datype='data'):
    if stopvar.is_set():
        raise Stopall()
    Ham1=UnpackHam(point,Hame,Vary)
    if fname in ['Powder','Calpowder']:
        fielda,intena=Powder(Ham1,Exp,M=Mr,graph=False)
    elif fname in ['Eresonant']:
        fielda,intena=Eresonant(Ham1,Exp,graph=False,table=False)
    if datype=='integral':
        intena=scii.cumulative_trapezoid(intena,fielda,initial=0)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    normi=np.linalg.norm(intena)
    if normi==0 or np.isnan(normi):
        return exper*1e6
    norme=np.linalg.norm(exper)
    enorm=exper/norme
    inorm=intena/normi
    return inorm-enorm

def Residualstracked(point,*args):
    resi=Residuals(point,*args)
    TRACK.record(float(np.sum(resi**2)),lambda: np.array(point,float))
    return resi

def LSquare1(Ham1,Expe,Vary,exper,maximal=1000,mode='p',Mr=70):
    '''
    Fitting adjutsment of the experimental data using the *scipy.optimize.least_squares* method. This case is for simple systems.
    
    Parameters
    ----------
    
    Ham1 : Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vary : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    
    spc : np.array
        Best adjusted spectrum using the Powder or Eresonant functions.
    '''
    TRACK.reset()
    if mode=='p':
        funtiona=Powder#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Eresonant#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    pointx,lowfron,hifron=Packtoscipy(Ham1,Vary)
    frontier=lowfron,hifron
    eps=1e-6
    pointx=np.clip(pointx,lowfron+eps,hifron-eps)
    try:
        res=leasts(fun=Residualstracked,args=(Ham1,Expe,exper,Vary,funcname,Mr),x0=pointx,bounds=frontier,method='trf',max_nfev=maximal,verbose=2,diff_step=1e-3,xtol=1e-14,ftol=1e-14,gtol=1e-14)
        bestone=res.x
        cost=res.cost
        success,message,iterations,J=res.success,res.message,res.nfev,res.jac          
    except Stopall:
        stopvar.clear()
        cost=float('nan')
        bestone=pointx
        success,message,iterations,J=False,"Stopped by user",None,None    
    bHam=UnpackHam(bestone,Ham1,Vary)
    print(f"Final cost: {cost:.5f}")
    print(f"Message: {message}")
    if Vary.g!=0.0:
        print(f'gx={bHam.g[0]} | gy={bHam.g[1]} | gz={bHam.g[2]}')
    if Vary.A!=0.0:
        print(f'Ax={bHam.A[0]} | Ay={bHam.A[1]} | Az={bHam.A[2]}')
    if Vary.Q!=0.0:
        print(f'Qx={bHam.Q[0]} | Qy={bHam.Q[1]} | Qz={bHam.Q[2]}')
    if Vary.D!=0.0:
        print(f'D={bHam.D[0]} | E={bHam.D[1]}')
    if np.any(Vary.Hpp):
        print(f'Hppg={bHam.Hpp[0]} | Hppl={bHam.Hpp[1]}')
    fitresult=Buildres(bestone,Ham1,Expe,exper,Vary,funcname,Mr,method='Least squares',J=J,success=success,message=message,iterations=iterations)
    return bHam,fitresult

def Packtoscipy2(Ham,Var):
    pointx=[]
    lowfron=[]
    hifron=[]
    for jer in range(0,len(Ham.Mulham)):
        vmi=Var.Mvary[jer]
        hmi=Ham.Mulham[jer]
        if vmi.g!=0.0:
            pointx.extend(hmi.g)
            lowfron.extend([vmi.g[0],vmi.g[2],vmi.g[4]])
            hifron.extend([vmi.g[1],vmi.g[3],vmi.g[5]])
        if vmi.A!=0.0:
            pointx.extend(hmi.A)
            lowfron.extend([vmi.A[0],vmi.A[2],vmi.A[4]])
            hifron.extend([vmi.A[1],vmi.A[3],vmi.A[5]])
        if vmi.Q!=0.0:
            pointx.extend(hmi.Q)
            lowfron.extend([vmi.Q[0],vmi.Q[2],vmi.Q[4]])
            hifron.extend([vmi.Q[1],vmi.Q[3],vmi.Q[5]])
        if vmi.D!=0.0:
            pointx.extend(hmi.D)
            lowfron.extend([vmi.D[0],vmi.D[2]])
            hifron.extend([vmi.D[1],vmi.D[3]])
        if np.any(vmi.Hpp):
            pointx.extend(hmi.Hpp)
            lowfron.extend([vmi.Hpp[0],vmi.Hpp[2]])
            hifron.extend([vmi.Hpp[1],vmi.Hpp[3]])
    pointx=np.array(pointx,dtype=float)
    lowfron=np.array(lowfron,dtype=float)
    hifron=np.array(hifron,dtype=float)
    return pointx,lowfron,hifron

def UnpackHam2(point,Hamer,Var):
    Ham=deepcopy(Hamer)
    idx=0
    for jer in range(len(Ham.Mulham)):
        vmi=Var.Mvary[jer]
        hmi=Ham.Mulham[jer]
        if vmi.g!=0.0:
            hmi.g=point[idx:idx+3]
            idx+=3
        if vmi.A!=0.0:
            hmi.A=point[idx:idx+3]
            idx+=3
        if vmi.Q!=0.0:
            hmi.Q=point[idx:idx+3]
            idx+=3
        if vmi.D!=0.0:
            hmi.D=point[idx:idx+2]
            idx+=2
        if np.any(vmi.Hpp):
            hmi.Hpp=point[idx:idx+2]
            idx+=2
    return Ham

def Residuals2(point,Hame,Exp,exper,Vary,fname,Mr=70,datype='data'):
    if stopvar.is_set():
        raise Stopall()
    Ham1=UnpackHam2(point,Hame,Vary)
    if fname in ['Mulpol']:
        fielda,intena=Mulpol(Ham1,Exp,M=Mr,graph=False)
    elif fname in ['Music']:
        fielda,intena=Music(Ham1,Exp,graph=False,table=False)
    if datype=='integral':
        intena=scii.cumulative_trapezoid(intena,fielda,initial=0)
        exper=scii.cumulative_trapezoid(exper,fielda,initial=0)
    normi=np.linalg.norm(intena)
    if normi==0 or np.isnan(normi):
        return exper*1e6
    norme=np.linalg.norm(exper)
    enorm=exper/norme
    inorm=intena/normi
    return inorm-enorm

def Residualstracked2(point,*args):
    resi=Residuals2(point,*args)
    TRACK.record(float(np.sum(resi**2)),lambda: np.array(point,float))
    return resi
    
def LSquare2(Ham1,Expe,Vary,exper,maximal=1000,mode='p',Mr=70):
    '''
    Fitting adjutsment of the experimental data using the *scipy.optimize.least_squares* method. This case is for multiple systems.
    
    Parameters
    ----------
    
    Ham1 : Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vary : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    
    spc : np.array
        Best adjusted spectrum using the Mulpol or Music functions.
    '''
    TRACK.reset()
    if mode=='p':
        funtiona=Mulpol#(Ham,Exp,graph=False)
    elif mode=='c':
        funtiona=Music#(Ham,Exp,graph=False,table=False)
    else:
        raise ValueError (f'Valid functions are powder (p) or cristal (c).')
    funcname=funtiona.__name__
    pointx,lowfron,hifron=Packtoscipy2(Ham1,Vary)
    frontier=lowfron,hifron
    eps=1e-6
    pointx=np.clip(pointx,lowfron+eps,hifron-eps)
    try:
        res=leasts(fun=Residualstracked2,args=(Ham1,Expe,exper,Vary,funcname,Mr),x0=pointx,bounds=frontier,method='trf',max_nfev=maximal,verbose=2,diff_step=1e-3,xtol=1e-14,ftol=1e-14,gtol=1e-14)
        bestone=res.x
        cost=res.cost
        success,message,iterations,J=res.success,res.message,res.nfev,res.jac       
    except Stopall:
        stopvar.clear()
        bestone=pointx
        cost=float('nan')
        success,message,iterations,J=False,"Stopped by user",None,None  
    bHam=UnpackHam2(bestone,Ham1,Vary)

    print(f"Final cost: {cost:.5f}")
    print(f"Message: {message}")
    for i in range(len(Ham1.Mulham)):
        print(f"--- System {i+1} ---")
        if Vary.Mvary[i].g!=0.0:
            print(f'gx={bHam.Mulham[i].g[0]:.4f} | gy={bHam.Mulham[i].g[1]:.4f} | gz={bHam.Mulham[i].g[2]:.4f}')
        if Vary.Mvary[i].A!=0.0:
            print(f'Ax={bHam.Mulham[i].A[0]:.4f} | Ay={bHam.Mulham[i].A[1]:.4f} | Az={bHam.Mulham[i].A[2]:.4f}')
        if Vary.Mvary[i].Q!=0.0:
            print(f'Qx={bHam.Mulham[i].Q[0]:.4f} | Qy={bHam.Mulham[i].Q[1]:.4f} | Qz={bHam.Mulham[i].Q[2]:.4f}')
        if Vary.Mvary[i].D!=0.0:
            print(f'D={bHam.Mulham[i].D[0]:.4f} | E={bHam.Mulham[i].D[1]:.4f}')
        if np.any(Vary.Mvary[i].Hpp):
            print(f'Hppg={bHam.Mulham[i].Hpp[0]:.4f} | Hppl={bHam.Mulham[i].Hpp[1]:.4f}')
    fitresult=Buildres(bestone,Ham1,Expe,exper,Vary,funcname,Mr,method='Least squares',J=J,success=success,message=message,iterations=iterations)
    return bHam,fitresult

def LSquare(Ham1,Expe,Vary,exper,maximal=1000,mode='p',Mr=70):
    '''
    Fitting adjutsment of the experimental data using the *scipy.optimize.least_squares* method. The Trust Region Reflective algorithm is use, with a tolerance for the change of variables (xtol, ftol, gtol) of 1e-10 and a maximun number of evaluations defined with the variable *maximal*. The documentation of the function can be consulted in https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html.
    
    Parameters
    ----------
    
    Ham1 : Class
        Container for the hamiltonian parameters.
    Expe : Class
        Container for the experimental conditions.
    Vary : Class
        Container for the range and parameters to vary.
    exper : np.array
        Experimental spectrum data to fit.
    maximal : int
        Max. number of evaluations of the function.
    mode : str
        Defines the sample type, 'p' for powder and 'c' for monocristal.
    Mr : int
        Number of divisions for the Delaunay grid.
        
    Returns
    -------
    
    Ham : Class
        Container for the fitted hamiltonian parameters.
    fitresult : Class 
        Container for the results of the Fitting function.
    '''
    if type(Ham1)==Multham:
         if Expe.Mexp[0].Points!=len(exper):
            Expe.Mexp[0].Points=len(exper)
         return LSquare2(Ham1,Expe,Vary,exper,maximal,mode,Mr)
    elif type(Ham1)==Hval:
         if Expe.Points!=len(exper):
            Expe.Points=len(exper)
         return LSquare1(Ham1,Expe,Vary,exper,maximal,mode,Mr)

def Fitting(Hamer,Exper,Vara,datexp):
    '''
    Wrap function for the data fitting process. Creates a interactive display where the user can select the fitting method, data and sample type, max. number of iterations and reference error. The function print the fitting results and spectrum, but they can also be access with epraya.Result.
    
    Parameters
    ----------
    
    Hamer : Class
        Container for the hamiltonian parameters.
    Exper : Class
        Container for the experimental conditions.
    Vara : Class
        Container for the range and parameters to vary.
    datexp : np.array
        Experimental spectrum data to fit.
        
    Example
    -------
    
    .. code-block:: python
    
       import epraya as epr
       B,spc=epr.Seek()
       Ham,Exp,Vary=epr.Start()
       Ham.S=1/2
       Ham.g=[2.003,2.003,2.003]
       Ham.Hpp=[0,0.3]
       Exp.Points=4096
       Exp.Frange=[B[0],B[-1]]
       Vary.g=[2.,2.2,2,2.2,2.,2.2]
       epr.Fitting(Ham,Exp,Vary,spc)

       
    .. image:: /_static/fit1.png
       :alt: Plot of the Fit1 function.
       :align: center
    
    .. image:: /_static/fit2.png
       :alt: Plot of the Fit2 function.
       :align: center
    '''        
    class OutputRedirector:
        def __init__(self, output_widget):
            self.output_widget=output_widget
        def write(self, string):
            self.output_widget.append_stdout(string)
        def flush(self):
            pass
    in_notebook=is_notebook()
    stopvar.clear()
    global result
    result.clear()
    lbl1=Label('Method:')
    wdg1=Dropdown(options=['Nelder-Mead','Genetic algorithm','Metropolis','Least squares'],index=2)
    frvar1=HBox([lbl1,wdg1])
    lbl2=Label('Data type:')
    wdg2=RadioButtons(options=['Spectrum','First Integral'],index=0)
    frvar2=HBox([lbl2,wdg2])
    lbl3=Label('Sample type:')
    wdg3=RadioButtons(options=['Powder','Cristal'],index=0)
    frvar3=HBox([lbl3,wdg3])
    lbl4=Label('Max. # of iterations:')
    wdg4=IntText(value=100,step=50,layout=Layout(width='200px'))
    frvar4=VBox([lbl4,wdg4])
    lbl5=Label('Max. error (10^x) [Input Exponent]:')
    wdg5=IntText(value=-10,step=1,layout=Layout(width='200px'))
    frvar5=VBox([lbl5,wdg5])
    lbl8=Label('Seed for distributions [Int]:')
    wdg8=IntText(value=451,step=1,layout=Layout(width='200px'))
    frvar8=VBox([lbl8,wdg8])
    lbl9=Label('Grid resolution:')
    wdg9=IntText(value=70,step=1,layout=Layout(width='200px'))
    frvar9=VBox([lbl9,wdg9])
    frvar6=Button(description='Start',button_style='success',icon='play',layout=Layout(border='2px solid blue'))
    frvar7=Button(description='Stop (Interrupt)',button_style='danger',icon='stop',layout=Layout(border='2px solid red'))
    tapts=HBox([frvar6,frvar7])
    outside=Output()
    partoeval=None
    def Evalfunc(b):
        nonlocal partoeval
        if partoeval is not None and partoeval.is_alive():
            with outside:
                print("\n[!] The last process is still running. Wait a moment and try again.")
            return
        stopvar.clear()
        with outside:
            outside.clear_output(wait=True)
            Chmet=wdg1.value
            cdtype=wdg2.value
            csample=wdg3.value
            numtr=wdg4.value
            erroreps=10**float(wdg5.value)
            seeds=wdg8.value
            mr=wdg9.value
            if cdtype=='Spectrum':
                cdtype='data'
            elif cdtype=='First Integral':
                cdtype='integral'
            if csample=='Powder':
                csample='p'
            elif csample=='Cristal':
                csample='c'
            
            def functiontorun():
                if in_notebook:
                    oldout=sys.stdout
                    sys.stdout=OutputRedirector(outside)
                try:
                    print(f'Starting process: {Chmet}...\n')
                    fitres=None
                    if Chmet=='Nelder-Mead':
                        Hamf,fitres=Nelder(Hamer,Exper,Vara,datexp,eps=erroreps,maximal=numtr,datype=cdtype,mode=csample,seed=seeds,Mr=mr)
                    elif Chmet=='Genetic algorithm':
                        Hamf,fitres=Genio(Hamer,Exper,Vara,datexp,eps=erroreps,maximal=numtr,datype=cdtype,mode=csample,seed=seeds,Mr=mr)
                    elif Chmet=='Metropolis':
                        Hamf,fitres=Metro(Hamer,Exper,Vara,datexp,maximal=numtr,datype=cdtype,mode=csample,seed=seeds,Mr=mr)
                    elif Chmet=='Least squares':
                        Hamf,fitres=LSquare(Hamer,Exper,Vara,datexp,maximal=numtr,mode=csample,Mr=mr)

                    global result
                    result['Ham']=Hamf
                    result['fit']=fitres
                    print(f"\nchi2 = {fitres.chi2:.5e} | chi2 residual = {fitres.redchi2:.5e}")
                    if fitres.paramerrors is not None:
                        print(f"Parameters error (1 sigma): {Formaterrors(fitres.paramerrors,fitres.paramlabels)}")

                    resultexper=fitres.spc
                    #To show the graph
                    fig=Figure(figsize=(8,10))
                    formatter=EngFormatter(sep='') 
                    canvas=FigureCanvasAgg(fig)
                    ax=fig.add_subplot(211)
                    if hasattr(Exper,'Mexp'):
                        Bla=np.linspace(Exper.Mexp[0].Frange[0],Exper.Mexp[0].Frange[1],Exper.Mexp[0].Points)
                    else:
                        Bla=np.linspace(Exper.Frange[0],Exper.Frange[1],Exper.Points)
                    ax.plot(Bla,datexp,color='blue',label='Data')
                    ax.yaxis.set_major_formatter(formatter)
                    ax.set_xlabel('Magnetic field [mT]')
                    ax.set_ylabel('Counts [A. U.]')
                    if resultexper is not None:
                        maximo=np.max(resultexper)
                        if maximo==0:
                            maximo=1
                        ax.plot(Bla,resultexper/np.max(resultexper)*np.max(datexp),color='red',label='Fit')
                    ax.legend()
                    ax.grid()
                    if in_notebook:
                        buf=io.BytesIO()
                        fig.savefig(buf,format='png',bbox_inches='tight')
                        buf.seek(0)
                        img=Image(data=buf.getvalue(),format='png')
                        outside.append_display_data(img)
                        buf.close()
                    else:
                        filename="fit_result.png"
                        fig.savefig(filename, format="png",bbox_inches="tight")
                        print(f"\n[+] Graph saved successfully as '{filename}'")
                    fig.clf()
                except Exception as e:
                    with outside:
                        print(f"\n[X] Error in function: {e}")
                finally:
                    if in_notebook:
                        sys.stdout=oldout
            partoeval=threading.Thread(target=functiontorun)
            partoeval.start()

    def Stopfunc(b):
        stopvar.set()
        with outside:
            print("\n[!] Stopping the process.")

    frvar6.on_click(Evalfunc)
    frvar7.on_click(Stopfunc)

    centerone=HBox([VBox([frvar1,frvar2,frvar3]),VBox([frvar4,frvar5]),VBox([frvar8,frvar9])])
    display(VBox([centerone,tapts,outside]))
    
    
def Selectvarian(J,redchi2,ztol=None,minw=0.3):
    '''
    Calculates the variance of the fit using the Jacobian matrix of the residuals. IT discart values where the variance cannot be calculated because it derivative is null or is lost in the numerical approximation, keeping the other ones. 
    
    The variance calculated in all cases is not conditional, to ensure its validity.
    
    Parameters
    ----------
    
    J : np.array
        Jacobian matrix of residuals.
    redchi2 : float
        Chi square for the calculation of the variance.
    ztol : float
        Reference value for parameter discart (column of the jacobian) if the variance cannot be calculated.
    minw : float
        Reference value for the weight (influence) of one parameter in the calculation of variance. If the weight is higher, the parameter cannot be discarted without affecting the variance validity.
        
    Returns
    -------
    variance : np.array
        Matrix of covariance of the fit.
    perr : np.array
        Values of the variance of the fitted parameters.
    identification: np.array
        Reference of the values that allows the calculation of the variance.
    
    '''
    p=J.shape[1]
    
    U,S,Vt=np.linalg.svd(J,full_matrices=False)
    Smax=S[0] if len(S)>0 else 0.0
    if ztol is None:
        ztol=max(J.shape)*np.finfo(float).eps 

    nulls=S<ztol*Smax if Smax>0 else np.ones_like(S,dtype=bool)
    #Calculates the inverse of the S matrix
    invS2=np.zeros_like(S)
    invS2[~nulls]=1.0/S[~nulls]**2

    variance=(Vt.T*invS2)@Vt*redchi2
    perr=np.sqrt(np.diag(variance))

    identification=np.ones(p,dtype=bool)
    if np.any(nulls):
        wg=np.abs(Vt[nulls,:])                    
        sick=np.any(wg**2>minw**2,axis=0)
        identification[sick]=False
        perr[sick]=np.nan

    return variance,perr,identification

In [6]:
import json
import time
import io
import contextlib
import pandas as pd
import scipy
import epraya as epr
from scipy.stats import qmc
import numpy as np

def Initialconf(Vary,samples=30,seed=42):
    '''
    Use Latin Hypercube Sampling to generate n initial configurations for the samples.
    '''
    low=[]
    high=[]

    if Vary.g!=0.0:
        low.extend([Vary.g[0],Vary.g[2],Vary.g[4]])
        high.extend([Vary.g[1],Vary.g[3],Vary.g[5]])

    if Vary.A!=0.0:
        low.extend([Vary.A[0],Vary.A[2],Vary.A[4]])
        high.extend([Vary.A[1],Vary.A[3],Vary.A[5]])

    if Vary.D!=0.0:
        low.extend([Vary.D[0],Vary.D[2]])
        high.extend([Vary.D[1],Vary.D[3]])

    if np.any(Vary.Hpp):
        low.extend([Vary.Hpp[0],Vary.Hpp[2]])
        high.extend([Vary.Hpp[1],Vary.Hpp[3]])
    dim=len(low)

    sampler=qmc.LatinHypercube(d=dim,seed=seed)
    samples=sampler.random(n=samples)
    initpoints=qmc.scale(samples,l_bounds=low,u_bounds=high)
    return initpoints
    
def Addenoise(spectrum,snr,seed=None):
    if seed is not None:
        np.random.seed(seed)
    hpp=np.ptp(spectrum)
    #Standard deviation sigma=Hpp/SNR
    sigma=hpp/float(snr)
    noise=np.random.normal(loc=0.0,scale=sigma,size=spectrum.shape)
    spectrum=spectrum+noise
    return spectrum,sigma

BIG=10**3
NGEN=100
runs={'Nelder': lambda H,E,V,y,m,s: Nelder(H,E,V,y,eps=1e-12,maximal=BIG,mode=m,seed=s,Mr=Mfit),
    'Metro': lambda H,E,V,y,m,s: Metro(H,E,V,y,maximal=BIG,mode=m,seed=s,Mr=Mfit),
    'LSQ': lambda H,E,V,y,m,s: LSquare(H,E,V,y,maximal=BIG,mode=m,Mr=Mfit),
    'Gen': lambda H,E,V,y,m,s: Genio(H,E,V,y,eps=1e-12,maximal=NGEN,mode=m,seed=s,Mr=Mfit),
}

#For powder simulation
Ham1,Exp1,Vary1=epr.Start()
Ham1.S=1
Ham1.I=1
Ham1.g=[2.003,3.001,2.5]
Ham1.A=[300,250,200]
Ham1.D=[200,600]
Ham1.Hpp=[10,5]
Exp1.Points=2048
Exp1.Freq=9.43
Exp1.Frange=[0,600]
Exp1.Temperature=300
Vary1.g=[1,4,1,4,1,4]
Vary1.A=[0,400,0,400,0,400]
Vary1.D=[0,500,200,800]
Vary1.Hpp=[0.5,30,0.5,30]
B1,spc1=epr.Powder(Ham1,Exp1,M=120,graph=False)
#For singlecrystal
Ham2,Exp2,Vary2=epr.Start()
Ham2.S=1
Ham2.I=1
Ham2.g=[2.003,3.001,2.5]
Ham2.A=[300,250,200]
Ham2.D=[200,600]
Ham2.Hpp=[10,5]
Exp2.Points=2048
Exp2.Freq=9.43
Exp2.Frange=[0,600]
Exp2.Temperature=300
Vary2.g=[1,4,1,4,1,4]
Vary2.A=[0,400,0,400,0,400]
Vary2.D=[0,500,200,800]
Vary2.Hpp=[0.5,30,0.5,30]
B2,spc2=epr.Eresonant(Ham2,Exp2,table=False,graph=False)
Nsamples=6
pinitipoints=Initialconf(Vary1,Nsamples,seed=123)
cinitipoints=Initialconf(Vary2,Nsamples,seed=124)
#Signal noise relation
snr=[20,10,5]                             
TRACK.budget=1000
ptrue=Packtoscipy(Ham1,Vary1)[0]
#For not strict criterium
Mfit=40
tol=scipy.stats.chi2.ppf(0.99,len(ptrue))   
for i,snrva in enumerate(snr):
    spc3,sigma=Addenoise(spc1,snrva,seed=100+i)
    sigrel=sigma/np.linalg.norm(spc3)
    chi2t=np.sum(Residuals(ptrue,Ham1,Exp1,spc3,Vary1,'Powder',Mfit)**2)
    #Minimun error achievable ar
    TRACK.target=chi2t+tol*sigrel**2
    print(f"SNR {snrva}: Chi2refernce={chi2t:.4e} | Target={TRACK.target:.4e}")
    for pidx,p0 in enumerate(pinitipoints):
        for name,run in runs.items():
            print("Method:",name)
            H0=UnpackHam(p0,Ham1,Vary1)
            t0=time.perf_counter()
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    run(H0,Exp1,Vary1,spc3,'p',451+pidx)
                #Stop before target
                status='stopped'              
            except TargetReached:
                #Reach target
                status='target'
            except BudgetReached:
                #Max number of iterations reached
                status='budget'
            last=TRACK.array() 
            row=dict(sample='p',method=name,snr=snrva,start=pidx,status=status,neval=int(TRACK.n),t=time.perf_counter()-t0,
                     best=float(TRACK.best),d=float((TRACK.best-chi2t)/sigrel**2),params=last[-1,3:].tolist())
            with open(f'target.jsonl','a') as f:
                f.write(json.dumps(row)+'\n')
            np.save(f'hist_{name}_{snrva}_{pidx}.npy',last)


df=pd.read_json('target2.jsonl',lines=True)
def resume(x):
    ok=x.status=='target'
    return pd.Series(dict(psuccess=ok.mean(),ERT=x.neval.sum()/ok.sum() if ok.any() else np.inf,mediansuccess=x.loc[ok,'neval'].median(),
                          dmedianfail=x.loc[~ok,'d'].median(),tiforeval=(x.t/x.neval).median()))
print(df.groupby(['method','snr']).apply(resume))

SNR 20: Chi2refernce=1.1434e-01 | Target=1.1553e-01
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
SNR 10: Chi2refernce=3.6762e-01 | Target=3.7111e-01
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
SNR 5: Chi2refernce=7.9041e-01 | Target=7.9806e-01
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: Gen
Method: Nelder
Method: Metro
Method: LSQ
Method: G

In [2]:

import json
import time
import io
import contextlib
import pandas as pd
import scipy
import epraya as epr
from scipy.stats import qmc
import numpy as np

def Initialconf(Vary,samples=30,seed=42):
    '''
    Use Latin Hypercube Sampling to generate n initial configurations for the samples.
    '''
    low=[]
    high=[]

    if Vary.g!=0.0:
        low.extend([Vary.g[0],Vary.g[2],Vary.g[4]])
        high.extend([Vary.g[1],Vary.g[3],Vary.g[5]])

    if Vary.A!=0.0:
        low.extend([Vary.A[0],Vary.A[2],Vary.A[4]])
        high.extend([Vary.A[1],Vary.A[3],Vary.A[5]])

    if Vary.D!=0.0:
        low.extend([Vary.D[0],Vary.D[2]])
        high.extend([Vary.D[1],Vary.D[3]])

    if np.any(Vary.Hpp):
        low.extend([Vary.Hpp[0],Vary.Hpp[2]])
        high.extend([Vary.Hpp[1],Vary.Hpp[3]])
    dim=len(low)

    sampler=qmc.LatinHypercube(d=dim,seed=seed)
    samples=sampler.random(n=samples)
    initpoints=qmc.scale(samples,l_bounds=low,u_bounds=high)
    return initpoints
    
def Addenoise(spectrum,snr,seed=None):
    if seed is not None:
        np.random.seed(seed)
    hpp=np.ptp(spectrum)
    #Standard deviation sigma=Hpp/SNR
    sigma=hpp/float(snr)
    noise=np.random.normal(loc=0.0,scale=sigma,size=spectrum.shape)
    spectrum=spectrum+noise
    return spectrum,sigma

BIG=10**3
NGEN=100
runs={'Nelder': lambda H,E,V,y,m,s: Nelder(H,E,V,y,eps=1e-12,maximal=BIG,mode=m,seed=s,Mr=Mfit),
    'Metro': lambda H,E,V,y,m,s: Metro(H,E,V,y,maximal=BIG,mode=m,seed=s,Mr=Mfit),
    'LSQ': lambda H,E,V,y,m,s: LSquare(H,E,V,y,maximal=BIG,mode=m,Mr=Mfit),
    'Gen': lambda H,E,V,y,m,s: Genio(H,E,V,y,eps=1e-12,maximal=NGEN,mode=m,seed=s,Mr=Mfit),
}

#For powder simulation
Ham1,Exp1,Vary1=epr.Start()
Ham1.S=1
Ham1.I=1
Ham1.g=[2.003,3.001,2.5]
Ham1.A=[300,250,200]
Ham1.D=[200,600]
Ham1.Hpp=[10,5]
Exp1.Points=2048
Exp1.Freq=9.43
Exp1.Frange=[0,600]
Exp1.Temperature=300
Vary1.g=[1,4,1,4,1,4]
Vary1.A=[0,400,0,400,0,400]
Vary1.D=[0,500,200,800]
Vary1.Hpp=[0.5,30,0.5,30]
B1,spc1=epr.Powder(Ham1,Exp1,M=120,graph=False)
#For singlecrystal
Ham2,Exp2,Vary2=epr.Start()
Ham2.S=1
Ham2.I=1
Ham2.g=[2.003,3.001,2.5]
Ham2.A=[300,250,200]
Ham2.D=[200,600]
Ham2.Hpp=[10,5]
Exp2.Points=2048
Exp2.Freq=9.43
Exp2.Frange=[0,600]
Exp2.Temperature=300
Vary2.g=[1,4,1,4,1,4]
Vary2.A=[0,400,0,400,0,400]
Vary2.D=[0,500,200,800]
Vary2.Hpp=[0.5,30,0.5,30]
B2,spc2=epr.Eresonant(Ham2,Exp2,table=False,graph=False)
Nsamples=6
pinitipoints=Initialconf(Vary1,Nsamples,seed=123)
cinitipoints=Initialconf(Vary2,Nsamples,seed=124)
#Signal noise relation
                       
TRACK.budget=1000
ptrue=Packtoscipy(Ham1,Vary1)[0]
#For not strict criterium
Mfit=40
tol=scipy.stats.chi2.ppf(0.99,len(ptrue))  
#Signal noise relation
snr=[30,20,10,5]                             
TRACK.budget=1000
ptrue=Packtoscipy(Ham2,Vary2)[0]
#For not strict criterium
tol=scipy.stats.chi2.ppf(0.99,len(ptrue))   
for i,snrva in enumerate(snr):
    spc4,sigma=Addenoise(spc2,snrva,seed=100+i)
    sigrel=sigma/np.linalg.norm(spc4)
    chi2t=np.sum(Residuals(ptrue,Ham2,Exp2,spc4,Vary2,'Eresonant')**2)
    #Minimun error achievable ar
    TRACK.target=chi2t+tol*sigrel**2
    print(f"SNR {snrva}: Chi2refernce={chi2t:.4e} | Target={TRACK.target:.4e}")
    for pidx,p0 in enumerate(cinitipoints):
        for name,run in runs.items():
            H0=UnpackHam(p0,Ham2,Vary2)
            t0=time.perf_counter()
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    run(H0,Exp2,Vary2,spc4,'c',451+pidx)
                #Stop before target
                status='stopped'              
            except TargetReached:
                #Reach target
                status='target'
            except BudgetReached:
                #Max number of iterations reached
                status='budget'
            last=TRACK.array() 
            row=dict(sample='c',method=name,snr=snrva,start=pidx,status=status,neval=int(TRACK.n),t=time.perf_counter()-t0,
                     best=float(TRACK.best),d=float((TRACK.best-chi2t)/sigrel**2),params=last[-1,3:].tolist())
            with open(f'ctarget.jsonl','a') as f:
                f.write(json.dumps(row)+'\n')
            np.save(f'histc_{name}_{snrva}_{pidx}.npy',last)


df=pd.read_json('ctarget.jsonl',lines=True)
def resume(x):
    ok=x.status=='target'
    return pd.Series(dict(psuccess=ok.mean(),ERT=x.neval.sum()/ok.sum() if ok.any() else np.inf,mediansuccess=x.loc[ok,'neval'].median(),
                          dmedianfail=x.loc[~ok,'d'].median(),tiforeval=(x.t/x.neval).median()))
print(df.groupby(['method','snr']).apply(resume))

SNR 30: Chi2refernce=9.3799e-02 | Target=9.4785e-02
SNR 20: Chi2refernce=2.0056e-01 | Target=2.0255e-01
SNR 10: Chi2refernce=4.9439e-01 | Target=4.9963e-01
SNR 5: Chi2refernce=1.0696e+00 | Target=1.0784e+00
            psuccess     ERT  mediansuccess   dmedianfail  tiforeval
method snr                                                          
Gen    5    0.166667  5796.0          796.0     41.309256   0.055319
       10   0.000000     inf            NaN     77.440635   0.056118
       20   0.000000     inf            NaN    169.044665   0.054002
       30   0.000000     inf            NaN    419.714470   0.050102
LSQ    5    0.166667  4157.0          243.0   2444.818319   0.058120
       10   0.166667  4211.0          177.0   6624.598335   0.051824
       20   0.166667  3194.0          155.0  21182.771609   0.054884
       30   0.166667  4438.0          496.0  44958.891079   0.051111
Metro  5    0.166667  5306.0          306.0   2132.411489   0.056758
       10   0.333333  2475.5      